# 16 · Финальный газовый прогноз v3: единственный train и bundle для backend (CPU)

**Вычисления:** CPU, **Accelerator = None**, интернет не нужен. Вход — выход ноутбука 14 (как у 15).

Что делает (решения зафиксированы до запуска, Drive 22–27, `docs/EXPORT_16_GAS_BUNDLE_V3_RU.md`):
- обучает **одну** модель — `G2_no_calendar` для `target_t4_gas_cross` (наблюдаемое строгое пересечение газа 1 %
  снизу вверх в [D+2; D+3)); признаки, гиперпараметры и выборка — ровно как в 15;
- рецепт `RECIPE` задаётся до запуска и не выбирается по результатам:
  - `refit_2025h1` — train до конца 2024, Platt на 2025 H1, однократный audit 2025 H2;
  - `fold_2024_exact` — точное воспроизведение fold_2024 из 15 (audit = 2024 H2);
- если в калибровке < 30 известных положительных — остановка **до** обучения (решение о рецепте — за Дашей);
- сверяет код с прогоном 15 (PR-AUC G2 fold_2024; для refit это отдельное сверочное обучение fold_2024 **до**
  финального) — в выборе не участвует, но расхождение > 0,001 **останавливает прогон без bundle** (fail-closed);
- упаковывает **исполняемый bundle без pickle**: `model.txt` (LightGBM), `bundle.json` (Platt, контракт, eligibility),
  `predictor.py`, код признаков 14 (`features_v3/`), контракт, синтетический пример, requirements, SHA-256;
- проверяет bundle, загруженный из файлов, против модели в памяти (parity), SHA всех файлов, обязательную дату среза,
  контракт и latency; при непройденной приёмке bundle удаляется. SHA-256 ZIP — отдельным файлом рядом с ZIP.

Это **proxy_score в shadow-режиме**: не вероятность пожара и не подтверждённый инцидент; `operational_ready = false`.
В выходных файлах нет ключей каналов и построчных прогнозов.


## Как запустить

1. **Add Input** → выход ноутбука 14 (тот же, что подключали к 15).
2. **Settings → Accelerator → None.**
3. Проверить `RECIPE` в следующей ячейке (одно значение — один прогон).
4. `MODE = "SMOKE"` — ≈1/10 каналов, несколько минут, результат не для backend. `MODE = "FULL"` → **Save Version → Save & Run All**.
5. Скачать из Output: `gas_cross_v3_bundle.zip`, `gas_cross_v3_bundle.zip.sha256`, `results_export_16_v3.json`,
   `summary_export_16_v3_ru.md`. Статус ≠ `completed` → bundle backend не передавать.


In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО ИЗМЕНЯТЬ
MODE = "FULL"             # "FULL" или "SMOKE"
RECIPE = "refit_2025h1"   # "refit_2025h1" или "fold_2024_exact" — решение до запуска, не по audit
print("Режим:", MODE, "| рецепт:", RECIPE)


In [ ]:
import os, platform, warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy, pandas, polars, sklearn, lightgbm
for m in (numpy, pandas, polars, sklearn, lightgbm):
    print(f"{m.__name__:10s}", m.__version__)
print("CPU       ", os.cpu_count())


## Встроенный код

Runtime 15, экспорт 16, predictor и код признаков 14 — те же файлы, что в репозитории (`ml/sensor_failure/`).


In [ ]:
# Исходный код ML (репозиторий на Kaggle не нужен): записывается в каталог и импортируется
SOURCES = {
"target_models_v3_runtime.py": "\"\"\"Ноутбук 15: сравнение моделей на целях панели v3 (T2a/T2b/T2c/T1a/T4) и typed-score контракт для бэкенда.\n\nПротокол зафиксирован до обучения (docs/EXPERIMENT_15_TARGETS_V3_RU.md):\n- только pre-2025, фолды fold_2023 / fold_2024 как в ноутбуке 05; 2021 (переход системы мониторинга, FAQ)\n  и 30 суток прогрева после него исключены;\n- purging по d_label_decision_end: метка обучающей строки решена до начала калибровки, калибровочной — до валидации;\n- модели: P0 априорная частота по типу датчика, R1 правило (недавние эпизоды), L1 логистическая регрессия,\n  G1 LightGBM, G2 LightGBM без календаря, G3 LightGBM без типа датчика/системы (проверка shortcut);\n- метрики: PR-AUC, ROC-AUC, Brier/ECE после Platt (калибровка на H1), бюджетная кривая top-k в сутки\n  с cooldown 72 ч и без; парный блочный бутстрэп по неделям для G1 против лучшего простого baseline;\n- выбор и evidence_level по заранее заданным gates; результат — typed-score контракт без построчных прогнозов.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport time\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nSCHEMA_VERSION = \"3.0\"\nTARGET_SPECS: dict[str, dict[str, Any]] = {\n    \"target_t2a_link_onset\": {\"tz\": \"T2a\", \"incident_type\": \"link_loss_onset\", \"family\": \"probability\",\n                              \"rule\": (\"f_link_onset_strict_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2b_link_sustained\": {\"tz\": \"T2b\", \"incident_type\": \"link_loss_sustained\",\n                                  \"family\": \"conditional_proxy_score\",\n                                  \"rule\": (\"f_link_completed_long_30d\", \"f_link_onset_strict_7d\"), \"competing\": \"competing_link_onset_d1\"},\n    \"target_t2c_power_onset\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_onset\", \"family\": \"probability\",\n                               \"rule\": (\"f_power_onset_strict_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t2c_power_sustained\": {\"tz\": \"T2c\", \"incident_type\": \"power_loss_sustained\",\n                                   \"family\": \"conditional_proxy_score\",\n                                   \"rule\": (\"f_power_completed_long_30d\", \"f_power_onset_strict_7d\"), \"competing\": \"competing_power_onset_d1\"},\n    \"target_t1a_tech_value\": {\"tz\": \"T1a\", \"incident_type\": \"measurement_fault_onset\",\n                              \"family\": \"probability\",\n                              \"rule\": (\"f_n_tech_like_30d\", \"f_n_undefined_measurement_30d\"), \"competing\": None},\n    \"target_t4_gas_cross\": {\"tz\": \"T4\", \"incident_type\": \"gas_threshold_cross\", \"family\": \"probability\",\n                            \"rule\": (\"f_gas_max_7d\", \"f_gas_band_n_7d\"), \"competing\": \"competing_gas_cross_d1\"},\n}\nCATEGORICAL = [\"link_state\", \"power_state\", \"тип_датчика\", \"тип_инж_системы\"]\nCALENDAR = [\"d_weekday\", \"d_month\"]\nCATALOGUE = [\"тип_датчика\", \"тип_инж_системы\"]\nFORBIDDEN_PREFIXES = (\"target_\", \"reason_\", \"competing_\", \"d_label\", \"d_target\", \"d_channel\", \"d_object\", \"d_cutoff\")\nFOLDS = [\n    {\"name\": \"fold_2023\", \"train_end\": date(2023, 1, 1), \"calibration\": (date(2023, 1, 1), date(2023, 7, 1)),\n     \"validation\": (date(2023, 7, 1), date(2024, 1, 1))},\n    {\"name\": \"fold_2024\", \"train_end\": date(2024, 1, 1), \"calibration\": (date(2024, 1, 1), date(2024, 7, 1)),\n     \"validation\": (date(2024, 7, 1), date(2025, 1, 1))},\n]\nLOAD_END = date(2025, 1, 1)\nEXCLUDED = (date(2021, 1, 1), date(2022, 1, 31))   # 2021 + 30 суток прогрева окон признаков\n# selectable, но исход неизвестен: такие строки ранжируются в бюджете (слоты в unknown считаются), но не входят в PR-AUC\nUNKNOWN_REASONS = [\"unobserved_window\", \"ambiguous_onset\", \"censored_duration\", \"unknown_previous_reading\"]\nMODELS = [\"P0_type_prior\", \"R1_rule\", \"L1_logistic\", \"G1_lightgbm\", \"G2_no_calendar\", \"G3_no_catalogue\"]\nBASELINES_FOR_GATE = [\"P0_type_prior\", \"R1_rule\"]\n\n\ndef make_config_targets(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if mode not in (\"FULL\", \"SMOKE\"):\n        raise ValueError(\"MODE должен быть FULL или SMOKE\")\n    smoke = mode == \"SMOKE\"\n    cfg = {\n        \"mode\": mode,\n        \"input_dir\": os.environ.get(\"LDT_KAGGLE_INPUT_DIR\", \"/kaggle/input\"),\n        \"output_dir\": os.environ.get(\"LDT_OUTPUT_DIR\", \"/kaggle/working\"),\n        \"targets\": list(TARGET_SPECS),\n        \"smoke_channel_share\": 10,\n        \"verify_sha256\": True,\n        \"random_seed\": 42,\n        \"lgbm_iterations\": 60 if smoke else 300,\n        \"max_neg_per_pos\": 30,\n        \"max_train_rows\": 300_000 if smoke else 2_000_000,\n        \"max_logistic_rows\": 100_000 if smoke else 500_000,\n        \"budgets_per_day\": [5, 10, 25, 50],\n        \"cooldown_days\": 3,\n        \"bootstrap_reps\": 20 if smoke else 100,\n        \"min_val_positives\": 30,\n        \"shortcut_tolerance\": 0.10,\n        \"ece_calibrated_max\": 0.02,      # probability_calibrated только при ECE ≤ 0,02 в обоих фолдах\n        \"min_known_share_selectable\": 0.5,  # g6: в валидации исход известен хотя бы у половины selectable строк\n        \"max_unknown_share_calibrated\": 0.10,  # probability_calibrated: unknown среди selectable ≤ 10 %\n        \"min_type_positives\": 30,        # срез по типам датчиков: типы с ≥ 30 положительными\n    }\n    cfg.update(overrides or {})\n    return cfg\n\n\n# ---------------------------------------------------------------------------\n# Данные\n# ---------------------------------------------------------------------------\ndef file_sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        while block := f.read(1 << 22):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef discover_panel(input_dir: Path, verify: bool) -> tuple[list[Path], dict[str, Any]]:\n    found = sorted(Path(input_dir).rglob(\"panel_manifest_v3.json\"))\n    if not found:\n        raise FileNotFoundError(\"panel_manifest_v3.json не найден: подключите выход ноутбука 14 как Input\")\n    manifest_path = found[0]\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"schema_version\") != SCHEMA_VERSION:\n        raise ValueError(f\"ожидалась схема {SCHEMA_VERSION}, в manifest {manifest.get('schema_version')}\")\n    parts = []\n    for part in manifest[\"parts\"]:\n        p = manifest_path.parent / part[\"file\"]\n        if verify and file_sha256(p) != part[\"sha256\"]:\n            raise ValueError(f\"SHA-256 части {part['file']} не совпадает с manifest\")\n        parts.append(p)\n    import polars as pl\n\n    rows = sum(int(pl.scan_parquet(p).select(pl.len()).collect().item()) for p in parts)\n    if rows != manifest[\"rows\"]:\n        raise ValueError(f\"строк в частях {rows:,}, в manifest {manifest['rows']:,}\")\n    return parts, manifest\n\n\ndef safe_features(manifest: dict[str, Any], columns: list[str]) -> list[str]:\n    # тип датчика/системы — справочные метаданные канала (есть в панели, но не в списке признаков manifest v3)\n    feats = [c for c in dict.fromkeys(list(manifest[\"feature_columns\"]) + CATALOGUE) if c in columns]\n    bad = [c for c in feats if c.startswith(FORBIDDEN_PREFIXES)]\n    if bad:\n        raise ValueError(f\"запрещённые признаки: {bad}\")\n    return feats\n\n\ndef load_target(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any],\n                load_end: date = LOAD_END) -> Any:\n    \"\"\"load_end: граница загрузки (15 — всегда 2025-01-01; финальный экспорт 16 задаёт свою по рецепту).\"\"\"\n    reason = \"reason_\" + target[len(\"target_\"):]\n    lf = pl.scan_parquet(parts)\n    lf = lf.filter((pl.col(\"d_cutoff_date\") < load_end) & (pl.col(\"d_label_decision_end\") <= load_end)\n                   & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    comp = TARGET_SPECS[target].get(\"competing\")\n    extra = [comp] if comp and comp in lf.collect_schema().names() else []\n    cols = [\"d_channel_key\", \"d_cutoff_date\", \"d_label_decision_end\", target, reason] + extra + features\n    return (lf.filter(pl.col(target).is_not_null() | pl.col(reason).is_in(UNKNOWN_REASONS)).select(cols)\n            .with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect())\n\n\nSTRESS_PERIOD = (date(2021, 1, 1), date(2022, 1, 1))   # переход системы мониторинга (FAQ): только stress-срез\n\n\ndef load_stress(pl: Any, parts: list[Path], target: str, features: list[str], cfg: dict[str, Any]) -> Any:\n    \"\"\"2021 — не участвует ни в обучении, ни в выборе; оценивается один раз моделями fold_2024 (GPT 20).\"\"\"\n    lf = pl.scan_parquet(parts).filter(pl.col(\"d_cutoff_date\").is_between(*STRESS_PERIOD, closed=\"left\")\n                                       & (pl.col(\"d_label_decision_end\") <= STRESS_PERIOD[1]) & pl.col(target).is_not_null())\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    numeric = [c for c in features if c not in CATEGORICAL]\n    return lf.select([target] + features).with_columns([pl.col(c).cast(pl.Float32) for c in numeric]).collect()\n\n\ndef coverage(pl: Any, parts: list[Path], targets: list[str], cfg: dict[str, Any]) -> dict[str, Any]:\n    lf = pl.scan_parquet(parts).filter((pl.col(\"d_cutoff_date\") < LOAD_END)\n                                       & ~pl.col(\"d_cutoff_date\").is_between(EXCLUDED[0], EXCLUDED[1], closed=\"left\"))\n    if cfg[\"mode\"] == \"SMOKE\":\n        lf = lf.filter(pl.col(\"d_channel_key\").hash(seed=7) % cfg[\"smoke_channel_share\"] == 0)\n    total = int(lf.select(pl.len()).collect().item())\n    out: dict[str, Any] = {\"rows_pre2025_without_2021\": total, \"targets\": {}}\n    for t in targets:\n        r = \"reason_\" + t[len(\"target_\"):]\n        g = lf.group_by(r).agg(pl.len()).collect()\n        reasons = {str(k): int(v) for k, v in g.iter_rows()}\n        labelled = reasons.get(\"positive\", 0) + reasons.get(\"negative\", 0) + reasons.get(\"negative_transient_only\", 0)\n        by_type = (lf.group_by(\"тип_датчика\").agg(pl.col(t).is_not_null().mean().alias(\"share\"), pl.len().alias(\"rows\"))\n                   .filter(pl.col(\"rows\") >= 1000).sort(\"rows\", descending=True).head(15).collect())\n        out[\"targets\"][t] = {\"labelled_rows\": labelled, \"labelled_share\": labelled / max(total, 1),\n                             \"reasons\": reasons,\n                             \"labelled_share_by_sensor_type\": {str(k or \"нет в справочнике\"): round(float(s), 4)\n                                                                for k, s, _ in by_type.iter_rows()}}\n    return out\n\n\ndef split_fold(pl: Any, frame: Any, fold: dict[str, Any]) -> tuple[Any, Any, Any]:\n    c0, c1 = fold[\"calibration\"]\n    v0, v1 = fold[\"validation\"]\n    d, e = pl.col(\"d_cutoff_date\"), pl.col(\"d_label_decision_end\")\n    train = frame.filter((d < fold[\"train_end\"]) & (e <= c0))\n    cal = frame.filter((d >= c0) & (d < c1) & (e <= v0))\n    val = frame.filter((d >= v0) & (d < v1) & (e <= v1))\n    return train, cal, val\n\n\ndef sample_train(pl: Any, train: Any, target: str, cfg: dict[str, Any]) -> Any:\n    pos = train.filter(pl.col(target) == 1)\n    neg = train.filter(pl.col(target) == 0)\n    n_neg = min(neg.height, max(pos.height * cfg[\"max_neg_per_pos\"], 1), max(cfg[\"max_train_rows\"] - pos.height, 1))\n    rate = n_neg / max(neg.height, 1)\n    neg = neg.sample(n=n_neg, seed=cfg[\"random_seed\"]) if n_neg < neg.height else neg\n    return pl.concat([pos.with_columns(pl.lit(1.0).alias(\"_w\")),\n                      neg.with_columns(pl.lit(1.0 / max(rate, 1e-9)).alias(\"_w\"))])\n\n\n# ---------------------------------------------------------------------------\n# Модели\n# ---------------------------------------------------------------------------\ndef _pandas(frame: Any, cols: list[str], levels: dict[str, list[str]]) -> Any:\n    import pandas as pd\n\n    x = pd.DataFrame({c: frame[c].to_numpy() for c in cols})\n    for c in cols:\n        if c in CATEGORICAL:\n            v = frame[c].fill_null(\"__NA__\").cast(str).to_numpy()\n            x[c] = pd.Categorical(v, categories=levels[c])\n    return x\n\n\nclass TypePrior:\n    def fit(self, train: Any, target: str, m: float = 50.0) -> \"TypePrior\":\n        import polars as pl\n\n        w = train[\"_w\"]\n        y = train[target].cast(float)\n        self.global_rate = float((w * y).sum() / w.sum())\n        g = train.with_columns(y.alias(\"_y\")).group_by(\"тип_датчика\").agg(\n            (pl.col(\"_w\") * pl.col(\"_y\")).sum().alias(\"p\"), pl.col(\"_w\").sum().alias(\"n\"))\n        self.rates = {k: (p + m * self.global_rate) / (n + m) for k, p, n in g.iter_rows()}\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return np.array([self.rates.get(k, self.global_rate) for k in frame[\"тип_датчика\"].to_list()], dtype=float)\n\n\ndef rule_score(frame: Any, spec: dict[str, Any]) -> np.ndarray:\n    a, b = spec[\"rule\"]\n    s = np.zeros(frame.height)\n    for col, w in ((a, 1.0), (b, 0.5)):\n        if col in frame.columns:\n            s = s + w * np.nan_to_num(frame[col].to_numpy().astype(float))\n    return s\n\n\nclass Logistic:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"Logistic\":\n        from sklearn.linear_model import LogisticRegression\n\n        if train.height > cfg[\"max_logistic_rows\"]:\n            train = train.sample(n=cfg[\"max_logistic_rows\"], seed=cfg[\"random_seed\"])\n        self.num = [c for c in features if c not in CATEGORICAL]\n        self.cat = {c: [str(v) for v in train[c].fill_null(\"__NA__\").cast(str).value_counts(sort=True)[c].head(50).to_list()]\n                    for c in features if c in CATEGORICAL}\n        x = self._x(train, fit=True)\n        self.model = LogisticRegression(C=1.0, max_iter=300, random_state=cfg[\"random_seed\"])\n        self.model.fit(x, train[target].to_numpy(), sample_weight=train[\"_w\"].to_numpy())\n        return self\n\n    def _x(self, frame: Any, fit: bool = False) -> np.ndarray:\n        num = np.column_stack([frame[c].to_numpy().astype(float) for c in self.num]) if self.num else np.zeros((frame.height, 0))\n        num = np.nan_to_num(num, nan=0.0, posinf=0.0, neginf=0.0)\n        num = np.sign(num) * np.log1p(np.abs(num))\n        if fit:\n            self.mu, self.sd = num.mean(0), num.std(0) + 1e-9\n        num = (num - self.mu) / self.sd\n        cats = [(frame[c].fill_null(\"__NA__\").cast(str).to_numpy()[:, None] == np.array(lv)[None, :]).astype(float)\n                for c, lv in self.cat.items()]\n        return np.hstack([num] + cats)\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(self._x(frame))[:, 1]\n\n\nclass LightGBM:\n    def fit(self, train: Any, target: str, features: list[str], cfg: dict[str, Any]) -> \"LightGBM\":\n        from lightgbm import LGBMClassifier\n\n        self.features = features\n        self.levels = {c: sorted(str(v) for v in train[c].fill_null(\"__NA__\").cast(str).unique().to_list())\n                       for c in features if c in CATEGORICAL}\n        self.model = LGBMClassifier(objective=\"binary\", n_estimators=int(cfg[\"lgbm_iterations\"]), learning_rate=0.05,\n                                    num_leaves=63, max_bin=63, min_child_samples=100, subsample=0.8, subsample_freq=1,\n                                    colsample_bytree=0.8, random_state=cfg[\"random_seed\"], n_jobs=-1, verbosity=-1)\n        cats = [c for c in features if c in CATEGORICAL]\n        self.model.fit(_pandas(train, features, self.levels), train[target].to_numpy(),\n                       sample_weight=train[\"_w\"].to_numpy(), categorical_feature=cats or \"auto\")\n        return self\n\n    def score(self, frame: Any) -> np.ndarray:\n        return self.model.predict_proba(_pandas(frame, self.features, self.levels))[:, 1]\n\n    def importance(self, top: int = 12) -> dict[str, float]:\n        gain = self.model.booster_.feature_importance(\"gain\")\n        total = float(gain.sum()) or 1.0\n        order = np.argsort(-gain)[:top]\n        return {self.features[i]: round(float(gain[i] / total), 4) for i in order}\n\n\n# ---------------------------------------------------------------------------\n# Метрики\n# ---------------------------------------------------------------------------\ndef _z(score: np.ndarray, is_probability: bool) -> np.ndarray:\n    if is_probability:\n        p = np.clip(score, 1e-6, 1 - 1e-6)\n        return np.log(p / (1 - p))\n    return np.log1p(np.maximum(score, 0))\n\n\ndef platt(raw_cal: np.ndarray, y_cal: np.ndarray, is_probability: bool) -> Any:\n    from sklearn.linear_model import LogisticRegression\n\n    if np.unique(y_cal).size < 2:\n        return None\n    m = LogisticRegression(max_iter=300)\n    m.fit(_z(raw_cal, is_probability).reshape(-1, 1), y_cal)\n    return m\n\n\ndef apply_platt(m: Any, raw: np.ndarray, is_probability: bool, prior: float) -> np.ndarray:\n    if m is None:\n        return np.full(len(raw), prior)\n    return m.predict_proba(_z(raw, is_probability).reshape(-1, 1))[:, 1]\n\n\ndef ece(y: np.ndarray, p: np.ndarray, bins: int = 10) -> float:\n    q = np.quantile(p, np.linspace(0, 1, bins + 1))\n    idx = np.clip(np.searchsorted(q, p, side=\"right\") - 1, 0, bins - 1)\n    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))\n\n\ndef budget_curve(days: np.ndarray, channels: np.ndarray, y: np.ndarray, score: np.ndarray,\n                 budgets: list[int], cooldown_days: int) -> dict[str, Any]:\n    \"\"\"top-k строк в сутки по всей selectable когорте; y: 1/0, −1 = исход неизвестен.\n    precision — нижняя граница (unknown = 0), precision_upper — верхняя (unknown = 1); recall — по известным положительным.\n    С cooldown канал не выдаётся повторно cooldown_days суток после алерта.\"\"\"\n    order = np.lexsort((-score, days))\n    d_sorted, c_sorted, y_sorted = days[order], channels[order], y[order]\n    starts = np.flatnonzero(np.r_[True, d_sorted[1:] != d_sorted[:-1]])\n    ends = np.r_[starts[1:], len(order)]\n    total_pos = int((y == 1).sum())\n    total_unk = int((y == -1).sum())\n    pos_sorted, unk_sorted = (y_sorted == 1).astype(int), (y_sorted == -1).astype(int)\n    out = {}\n    for k in budgets:\n        tp = n = unk = 0\n        tp_cd = n_cd = unk_cd = 0\n        last: dict[Any, int] = {}\n        for s, e in zip(starts, ends):\n            tp += int(pos_sorted[s:min(e, s + k)].sum())\n            unk += int(unk_sorted[s:min(e, s + k)].sum())\n            n += min(e - s, k)\n            day = int(d_sorted[s])\n            taken = 0\n            for i in range(s, e):\n                if taken >= k:\n                    break\n                ch = c_sorted[i]\n                if ch in last and day - last[ch] < cooldown_days:\n                    continue\n                last[ch] = day\n                taken += 1\n                tp_cd += int(pos_sorted[i])\n                unk_cd += int(unk_sorted[i])\n            n_cd += taken\n        out[str(k)] = {\"precision\": tp / max(n, 1), \"precision_upper\": (tp + unk) / max(n, 1), \"unknown_alerts\": unk,\n                       \"recall\": tp / max(total_pos, 1), \"alerts\": n,\n                       # recall по GPT 16: нижняя — все неизвестные вне бюджета положительные, верхняя — все выданные unknown положительные\n                       \"recall_low\": tp / max(total_pos + total_unk - unk, 1), \"recall_up\": (tp + unk) / max(total_pos + unk, 1),\n                       \"precision_cooldown\": tp_cd / max(n_cd, 1), \"precision_upper_cooldown\": (tp_cd + unk_cd) / max(n_cd, 1),\n                       \"unknown_alerts_cooldown\": unk_cd, \"recall_cooldown\": tp_cd / max(total_pos, 1),\n                       \"recall_low_cooldown\": tp_cd / max(total_pos + total_unk - unk_cd, 1),\n                       \"recall_up_cooldown\": (tp_cd + unk_cd) / max(total_pos + unk_cd, 1),\n                       \"alerts_cooldown\": n_cd}\n    return out\n\n\ndef evaluate(y: np.ndarray, raw: np.ndarray, cal: np.ndarray, budget_frame: tuple[np.ndarray, ...],\n             cfg: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"Ранжирующие метрики и калибровка — по строкам с известным исходом; бюджет — по всей selectable когорте.\"\"\"\n    days, channels, y_all, raw_all = budget_frame\n    from sklearn.metrics import average_precision_score, roc_auc_score\n\n    ok = np.unique(y).size == 2\n    return {\n        \"pr_auc\": float(average_precision_score(y, raw)) if ok else None,\n        \"roc_auc\": float(roc_auc_score(y, raw)) if ok else None,\n        \"brier\": float(np.mean((cal - y) ** 2)),\n        \"ece10\": ece(y, cal),\n        \"budget\": budget_curve(days, channels, y_all, raw_all, cfg[\"budgets_per_day\"], cfg[\"cooldown_days\"]),\n    }\n\n\ndef by_type_pr_auc(y: np.ndarray, raw: np.ndarray, types: np.ndarray, min_pos: int) -> dict[str, float]:\n    from sklearn.metrics import average_precision_score\n\n    out = {}\n    for t in np.unique(types):\n        m = types == t\n        if y[m].sum() >= min_pos and y[m].min() != y[m].max():\n            out[str(t)] = float(average_precision_score(y[m], raw[m]))\n    return out\n\n\ndef paired_bootstrap(y: np.ndarray, a: np.ndarray, b: np.ndarray, weeks: np.ndarray, reps: int, seed: int) -> dict[str, Any]:\n    \"\"\"PR-AUC(a) − PR-AUC(b), блоки = календарные недели, одни и те же выборки для обеих моделей.\"\"\"\n    from sklearn.metrics import average_precision_score\n\n    uniq, inv = np.unique(weeks, return_inverse=True)\n    rng = np.random.default_rng(seed)\n    diffs = []\n    for _ in range(reps):\n        w = np.bincount(rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[inv].astype(float)\n        m = w > 0\n        if y[m].min() == y[m].max():\n            continue\n        diffs.append(average_precision_score(y[m], a[m], sample_weight=w[m])\n                     - average_precision_score(y[m], b[m], sample_weight=w[m]))\n    d = np.array(diffs)\n    if not len(d):\n        return {\"mean\": None, \"ci95_low\": None, \"ci95_high\": None, \"reps\": 0}\n    return {\"mean\": float(d.mean()), \"ci95_low\": float(np.percentile(d, 2.5)),\n            \"ci95_high\": float(np.percentile(d, 97.5)), \"reps\": int(len(d))}\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_target(pl: Any, frame: Any, target: str, features: list[str], cfg: dict[str, Any], log: Any,\n               stress: Any = None) -> dict[str, Any]:\n    spec = TARGET_SPECS[target]\n    res: dict[str, Any] = {\"folds\": {}}\n    for fold in FOLDS:\n        train, cal, val_all = split_fold(pl, frame, fold)\n        known = pl.col(target).is_not_null()\n        train, cal, val = train.filter(known), cal.filter(known), val_all.filter(known)\n        n = {\"train\": train.height, \"calibration\": cal.height, \"validation\": val.height,\n             \"train_pos\": int(train[target].sum()), \"calibration_pos\": int(cal[target].sum()),\n             \"validation_pos\": int(val[target].sum()), \"validation_selectable\": val_all.height,\n             \"validation_unknown\": val_all.height - val.height}\n        fr: dict[str, Any] = {\"rows\": n, \"models\": {}}\n        res[\"folds\"][fold[\"name\"]] = fr\n        if min(n[\"train_pos\"], n[\"calibration_pos\"]) < 5 or n[\"validation_pos\"] < 1:\n            fr[\"skipped\"] = \"слишком мало положительных для обучения/калибровки\"\n            continue\n        tr = sample_train(pl, train, target, cfg)\n        y_cal, y_val = cal[target].to_numpy().astype(int), val[target].to_numpy().astype(int)\n        days = np.array([(d - date(2019, 1, 1)).days for d in val[\"d_cutoff_date\"].to_list()])\n        weeks = days // 7\n        days_all = np.array([(d - date(2019, 1, 1)).days for d in val_all[\"d_cutoff_date\"].to_list()])\n        chans_all = val_all[\"d_channel_key\"].to_numpy()\n        y_all = val_all[target].fill_null(-1).to_numpy().astype(int)\n        known_mask = y_all >= 0\n        prior = float(train[target].mean())\n        raws: dict[str, np.ndarray] = {}\n        fitted: dict[str, Any] = {}\n        no_cal = [c for c in features if c not in CALENDAR]\n        no_cat = [c for c in features if c not in CATALOGUE]\n        for name in MODELS:\n            t0 = time.time()\n            if name == \"P0_type_prior\":\n                m = TypePrior().fit(tr, target)\n                is_p = True\n            elif name == \"R1_rule\":\n                m, is_p = None, False\n            elif name == \"L1_logistic\":\n                m, is_p = Logistic().fit(tr, target, features, cfg), True\n            else:\n                feats = {\"G1_lightgbm\": features, \"G2_no_calendar\": no_cal, \"G3_no_catalogue\": no_cat}[name]\n                m, is_p = LightGBM().fit(tr, target, feats, cfg), True\n            score = (lambda f: rule_score(f, spec)) if m is None else m.score\n            raw_cal, raw_all = score(cal), score(val_all)\n            raw_val = raw_all[known_mask]\n            cal_model = platt(raw_cal, y_cal, is_p)\n            p_val = apply_platt(cal_model, raw_val, is_p, prior)\n            raws[name] = raw_val\n            fitted[name] = m\n            fr[\"models\"][name] = evaluate(y_val, raw_val, p_val, (days_all, chans_all, y_all, raw_all), cfg)\n            fr[\"models\"][name][\"fit_s\"] = round(time.time() - t0, 1)\n        fr[\"prevalence_validation\"] = float(y_val.mean())\n        fr[\"unknown_share_selectable\"] = n[\"validation_unknown\"] / max(n[\"validation_selectable\"], 1)\n        comp = spec.get(\"competing\")\n        if comp and comp in val.columns:\n            from sklearn.metrics import average_precision_score\n            # C5 / GPT 16: estimand «любой onset в D+2» против «первого» (onset в D+1 — отдельный исход), строки не удаляются\n            c = val[comp].fill_null(False).to_numpy().astype(bool)\n            y_first = (y_val == 1) & ~c\n            fr[\"competing_d1\"] = {\"share_of_rows\": float(c.mean()), \"share_of_positives\": float(c[y_val == 1].mean()) if y_val.sum() else None,\n                                  \"pr_auc_first_onset\": {k: (float(average_precision_score(y_first, raws[k]))\n                                                             if y_first.any() else None) for k in MODELS}}\n        types = val[\"тип_датчика\"].fill_null(\"нет в справочнике\").to_numpy()\n        fr[\"by_sensor_type_pr_auc\"] = {k: by_type_pr_auc(y_val, raws[k], types, cfg[\"min_type_positives\"]) for k in MODELS}\n        fr[\"importance_G1\"] = fitted[\"G1_lightgbm\"].importance()\n        best_base = max(BASELINES_FOR_GATE, key=lambda k: fr[\"models\"][k][\"pr_auc\"] or -1)\n        fr[\"best_simple_baseline\"] = best_base\n        for cand in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\"):\n            fr[f\"bootstrap_{cand}_vs_{best_base}\"] = paired_bootstrap(\n                y_val, raws[cand], raws[best_base], weeks, cfg[\"bootstrap_reps\"], cfg[\"random_seed\"])\n        log(f\"  {fold['name']}: val {n['validation']:,} строк, 1: {n['validation_pos']:,}; PR-AUC \"\n            + \", \".join(f\"{k.split('_')[0]} {fr['models'][k]['pr_auc']:.3f}\" for k in MODELS if fr['models'][k]['pr_auc'] is not None))\n    res[\"decision\"] = decide(res, cfg)\n    last = res[\"folds\"].get(FOLDS[-1][\"name\"], {})\n    if stress is not None and stress.height and \"skipped\" not in last and last.get(\"models\"):\n        from sklearn.metrics import average_precision_score\n        y_s = stress[target].to_numpy().astype(int)\n        out = {\"rows\": stress.height, \"positives\": int(y_s.sum()), \"prevalence\": float(y_s.mean()), \"pr_auc\": {}}\n        if 0 < y_s.sum() < len(y_s):\n            for name, m in fitted.items():\n                raw = rule_score(stress, spec) if m is None else m.score(stress)\n                out[\"pr_auc\"][name] = float(average_precision_score(y_s, raw))\n        res[\"stress_2021\"] = out\n    return res\n\n\ndef decide(res: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    folds = list(res[\"folds\"].values())\n    gates: dict[str, Any] = {}\n    gates[\"g1_enough_validation_events\"] = all(f[\"rows\"][\"validation_pos\"] >= cfg[\"min_val_positives\"] and \"skipped\" not in f\n                                               for f in folds)\n    if not gates[\"g1_enough_validation_events\"]:\n        return {\"gates\": gates, \"selected\": None, \"evidence_level\": \"insufficient_events\",\n                \"abstain_reason\": \"too_few_validation_events\"}\n\n    def beats(model: str) -> bool:\n        return all((f[f\"bootstrap_{model}_vs_{f['best_simple_baseline']}\"][\"ci95_low\"] or -1) > 0 for f in folds)\n\n    def ratio(model: str) -> float:\n        return min(f[\"models\"][model][\"pr_auc\"] / max(f[\"models\"][\"G1_lightgbm\"][\"pr_auc\"], 1e-12) for f in folds)\n\n    tol = 1 - cfg[\"shortcut_tolerance\"]\n    gates[\"g2_G1_beats_simple_baselines_both_folds\"] = beats(\"G1_lightgbm\")\n    gates[\"g3_no_calendar_shortcut\"] = ratio(\"G2_no_calendar\") >= tol\n    gates[\"g4_no_catalogue_shortcut\"] = ratio(\"G3_no_catalogue\") >= tol\n    base_means = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in BASELINES_FOR_GATE}\n\n    def stable(model: str) -> bool | None:\n        \"\"\"≥ 2/3 типов датчиков с поддержкой: модель лучше лучшего простого baseline в каждом фолде.\n        Неприменимо (None), если в фолдах меньше двух типов с поддержкой (например, газовая цель T4).\"\"\"\n        applicable = False\n        for f in folds:\n            base = f[\"by_sensor_type_pr_auc\"][f[\"best_simple_baseline\"]]\n            mine = f[\"by_sensor_type_pr_auc\"][model]\n            common = [k for k in mine if k in base]\n            if len(common) < 2:\n                continue\n            applicable = True\n            if sum(mine[k] > base[k] for k in common) < 2 / 3 * len(common):\n                return False\n        return True if applicable else None\n\n    gates[\"g5_logistic_beats_simple_baselines_both_folds\"] = beats(\"L1_logistic\")\n    mean_pr = {k: float(np.mean([f[\"models\"][k][\"pr_auc\"] for f in folds])) for k in (\"G1_lightgbm\", \"G2_no_calendar\", \"L1_logistic\")}\n    if gates[\"g2_G1_beats_simple_baselines_both_folds\"]:\n        # если без календаря модель тоже побеждает baseline, а календарь даёт подозрительно много — берём G2\n        selected = \"G2_no_calendar\" if (not gates[\"g3_no_calendar_shortcut\"] and beats(\"G2_no_calendar\")) else \"G1_lightgbm\"\n        # при равенстве (±0,005 PR-AUC) предпочитаем более простую и устойчивую логистическую регрессию\n        if gates[\"g5_logistic_beats_simple_baselines_both_folds\"] and mean_pr[\"L1_logistic\"] >= mean_pr[selected] - 0.005:\n            selected = \"L1_logistic\"\n        level = \"validated_proxy_model\"\n    elif gates[\"g5_logistic_beats_simple_baselines_both_folds\"]:\n        selected, level = \"L1_logistic\", \"validated_proxy_model\"\n    else:\n        selected = max(base_means, key=base_means.get)\n        level = \"baseline_only\"\n    gates[\"g6_known_outcome_share\"] = all(1 - f[\"unknown_share_selectable\"] >= cfg[\"min_known_share_selectable\"] for f in folds)\n    gates[\"g7_stable_across_sensor_types\"] = stable(selected) if level == \"validated_proxy_model\" else None\n    return {\"gates\": gates, \"selected\": selected, \"evidence_level\": level, \"abstain_reason\": None,\n            \"simple_baseline_mean_pr_auc\": base_means}\n\n\ndef score_kind(spec: dict[str, Any], decision: dict[str, Any], folds: dict[str, Any], cfg: dict[str, Any]) -> str:\n    \"\"\"Что отдаёт сервис: калиброванную вероятность обещаем только при подтверждённой модели и ECE ≤ порога.\"\"\"\n    if decision[\"evidence_level\"] == \"insufficient_events\":\n        return \"abstain\"\n    if spec[\"family\"] == \"conditional_proxy_score\":\n        return \"conditional_proxy_score\"  # цензура длительности: на всю популяцию не калибруем (согласовано 10/Q1)\n    sel = decision[\"selected\"]\n    eces = [f[\"models\"][sel][\"ece10\"] for f in folds.values() if sel in f.get(\"models\", {})]\n    unknown = [f.get(\"unknown_share_selectable\", 1.0) for f in folds.values()]\n    # ECE на наблюдаемых строках не доказывает калибровку для всех каналов: требуем ещё малую долю unknown (GPT 14)\n    if (decision[\"evidence_level\"] == \"validated_proxy_model\" and eces and max(eces) <= cfg[\"ece_calibrated_max\"]\n            and max(unknown) <= cfg[\"max_unknown_share_calibrated\"]):\n        return \"probability_calibrated\"\n    return \"proxy_score\"\n\n\ndef backend_contract(results: dict[str, Any], cov: dict[str, Any], cfg: dict[str, Any]) -> dict[str, Any]:\n    out = {\"schema_version\": SCHEMA_VERSION, \"mode\": cfg[\"mode\"],\n           \"window\": \"метка: события в [D+2; D+3) по признакам на конец D; решение метки не позже D+4\",\n           \"cooldown_days\": cfg[\"cooldown_days\"], \"targets\": [],\n           \"data_quality_flags\": [{\"tz_item\": \"T1b\", \"incident_type\": \"clock_error_1970\", \"score_kind\": \"data_quality_flag\",\n                                   \"source\": \"значение 01.01.1970 в журнале (FAQ: «сбилась дата»); признаки f_n_clock_1970_*\",\n                                   \"not_a_failure_label\": True}],\n           \"evidence_levels\": {\"validated_proxy_model\": \"модель лучше простых baseline в обоих фолдах (95% CI)\",\n                               \"baseline_only\": \"отдаётся скор простого правила/приора\",\n                               \"insufficient_events\": \"мало событий — abstain\"}}\n    for t, r in results.items():\n        spec = TARGET_SPECS[t]\n        d = r[\"decision\"]\n        last = r[\"folds\"].get(\"fold_2024\", {})\n        sel = d.get(\"selected\")\n        m = last.get(\"models\", {}).get(sel, {}) if sel else {}\n        reasons = cov[\"targets\"][t][\"reasons\"]\n        abstain = {k: v for k, v in sorted(reasons.items(), key=lambda kv: -kv[1])\n                   if k not in (\"positive\", \"negative\", \"negative_transient_only\")}\n        entry = {\n            \"target\": t, \"tz_item\": spec[\"tz\"], \"incident_type\": spec[\"incident_type\"],\n            \"score_kind\": score_kind(spec, d, r[\"folds\"], cfg),\n            \"evidence_level\": d[\"evidence_level\"], \"selected_model\": sel,\n            \"coverage_labelled_share\": round(cov[\"targets\"][t][\"labelled_share\"], 4),\n            \"abstain_reasons_rows\": abstain,\n            \"validation_fold_2024\": ({\"pr_auc_on_known\": m.get(\"pr_auc\"), \"prevalence_on_known\": last.get(\"prevalence_validation\"),\n                                      \"unknown_share_selectable\": last.get(\"unknown_share_selectable\"),\n                                      \"brier\": m.get(\"brier\"), \"ece10\": m.get(\"ece10\"),\n                                      \"budget_25_per_day\": m.get(\"budget\", {}).get(\"25\")} if m else None),\n            \"not_a_confirmed_failure\": True,\n        }\n        if spec[\"family\"] == \"conditional_proxy_score\":\n            lab = cov[\"targets\"][t][\"labelled_rows\"]\n            cen = reasons.get(\"censored_duration\", 0)\n            entry[\"censoring\"] = {\"censored_rows\": cen, \"censored_share_of_decidable\": cen / max(lab + cen, 1),\n                                  \"note\": \"метрики — по нецензурированным строкам; для всей популяции это bounds, не оценка\"}\n        out[\"targets\"].append(entry)\n    return out\n\n\ndef run_targets_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t_start = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    parts, manifest = discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    columns = pl.scan_parquet(parts).collect_schema().names()\n    features = safe_features(manifest, columns)\n    log(f\"панель: {manifest['rows']:,} строк, частей {len(parts)}, признаков {len(features)}; режим {cfg['mode']}\")\n    cov = coverage(pl, parts, cfg[\"targets\"], cfg)\n    results = {}\n    for t in cfg[\"targets\"]:\n        log(f\"{t}:\")\n        frame = load_target(pl, parts, t, features, cfg)\n        stress = load_stress(pl, parts, t, features, cfg)\n        results[t] = run_target(pl, frame, t, features, cfg, log, stress=stress)\n        del stress\n        del frame\n    contract = backend_contract(results, cov, cfg)\n    status = \"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\"\n    payload = {\"schema_version\": SCHEMA_VERSION, \"status\": status, \"mode\": cfg[\"mode\"],\n               \"panel_manifest\": {\"rows\": manifest[\"rows\"], \"taxonomy_sha256\": manifest[\"taxonomy_sha256\"],\n                                  \"parts_sha256\": [p[\"sha256\"] for p in manifest[\"parts\"]],\n                                  \"panel_config\": manifest[\"panel_config\"]},\n               \"protocol\": {\"folds\": FOLDS,\n                            \"excluded_cutoff_dates\": [str(x) for x in EXCLUDED], \"load_end_exclusive\": str(LOAD_END),\n                            \"features\": features, \"models\": MODELS},\n               \"config\": {k: v for k, v in cfg.items() if k not in (\"input_dir\", \"output_dir\")},\n               \"coverage\": cov, \"targets\": results, \"runtime_s\": round(time.time() - t_start, 1)}\n    (out / \"results_targets_v3.json\").write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding=\"utf-8\")\n    (out / \"backend_typed_scores_v3.json\").write_text(json.dumps(contract, ensure_ascii=False, indent=2, default=str),\n                                                      encoding=\"utf-8\")\n    (out / \"summary_targets_v3_ru.md\").write_text(summary_markdown(payload, contract), encoding=\"utf-8\")\n    return {\"status\": status, \"runtime_s\": payload[\"runtime_s\"],\n            \"decisions\": {t: r[\"decision\"][\"evidence_level\"] for t, r in results.items()}}\n\n\ndef _f(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\ndef summary_markdown(payload: dict[str, Any], contract: dict[str, Any]) -> str:\n    L = [f\"# Ноутбук 15 — модели на целях панели v3 ({payload['mode']})\", \"\"]\n    if payload[\"mode\"] == \"SMOKE\":\n        L += [\"**SMOKE:** часть каналов, мало деревьев — цифры не сравнимы с FULL.\", \"\"]\n    L += [\"Метки — наблюдаемые события журнала, не подтверждённые отказы. 2021 и январь 2022 исключены; \"\n          \"2025+ не загружается.\", \"\", \"## Итог по целям\", \"\",\n          \"| цель | ТЗ | evidence_level | score_kind | модель | покрытие строк | PR-AUC 2023 | PR-AUC 2024 | unknown 2024 | precision@25/сут (cooldown) 2024, границы |\",\n          \"|---|---|---|---|---|---|---|---|---|---|\"]\n    for c in contract[\"targets\"]:\n        r = payload[\"targets\"][c[\"target\"]]\n        sel = c[\"selected_model\"]\n        pr = [_f(r[\"folds\"][f].get(\"models\", {}).get(sel, {}).get(\"pr_auc\")) if sel else \"—\" for f in (\"fold_2023\", \"fold_2024\")]\n        v = c[\"validation_fold_2024\"] or {}\n        b = v.get(\"budget_25_per_day\") or {}\n        L.append(f\"| `{c['target']}` | {c['tz_item']} | **{c['evidence_level']}** | {c['score_kind']} | {sel or '—'} | \"\n                 f\"{c['coverage_labelled_share']:.3f} | {pr[0]} | {pr[1]} | {_f(v.get('unknown_share_selectable'))} | \"\n                 f\"{_f(b.get('precision_cooldown'))}–{_f(b.get('precision_upper_cooldown'))} |\")\n    for t, r in payload[\"targets\"].items():\n        L += [\"\", f\"## `{t}`\", \"\"]\n        for fname, f in r[\"folds\"].items():\n            L += [f\"**{fname}**: train {f['rows']['train']:,} (1: {f['rows']['train_pos']:,}), \"\n                  f\"validation {f['rows']['validation']:,} (1: {f['rows']['validation_pos']:,})\", \"\"]\n            if \"skipped\" in f:\n                L += [f\"пропущен: {f['skipped']}\", \"\"]\n                continue\n            L += [f\"selectable в валидации: {f['rows']['validation_selectable']:,}, исход неизвестен: \"\n                  f\"{f['rows']['validation_unknown']:,} ({f['unknown_share_selectable']:.1%}). PR-AUC/ROC/Brier/ECE — по известным; \"\n                  \"P — нижняя–верхняя граница по всей selectable когорте, R — по известным положительным.\", \"\",\n                  \"| модель | PR-AUC | ROC-AUC | Brier | ECE | P@10 | R@10 | P@25 cd | R@25 cd | слотов в unknown @25 cd |\",\n                  \"|---|---|---|---|---|---|---|---|---|---|\"]\n            for m, x in f[\"models\"].items():\n                b10, b25 = x[\"budget\"].get(\"10\", {}), x[\"budget\"].get(\"25\", {})\n                L.append(f\"| {m} | {_f(x['pr_auc'])} | {_f(x['roc_auc'])} | {_f(x['brier'], 4)} | {_f(x['ece10'], 4)} | \"\n                         f\"{_f(b10.get('precision'))}–{_f(b10.get('precision_upper'))} | {_f(b10.get('recall'))} | \"\n                         f\"{_f(b25.get('precision_cooldown'))}–{_f(b25.get('precision_upper_cooldown'))} | \"\n                         f\"{_f(b25.get('recall_cooldown'))} | {b25.get('unknown_alerts_cooldown', 0):,} |\")\n            bb = f[\"best_simple_baseline\"]\n            bs = f[f\"bootstrap_G1_lightgbm_vs_{bb}\"]\n            bl = f[f\"bootstrap_L1_logistic_vs_{bb}\"]\n            L += [\"\", f\"G1 − {bb}: ΔPR-AUC {_f(bs['mean'])} [{_f(bs['ci95_low'])}; {_f(bs['ci95_high'])}], повторов {bs['reps']}; \"\n                  f\"L1 − {bb}: {_f(bl['mean'])} [{_f(bl['ci95_low'])}; {_f(bl['ci95_high'])}].\",\n                  \"Топ признаков G1: \" + \", \".join(f\"{k} {v:.2f}\" for k, v in list(f[\"importance_G1\"].items())[:6]), \"\"]\n            b50 = f[\"models\"][\"G1_lightgbm\"][\"budget\"].get(\"50\", {})\n            L += [f\"Top-50/сутки, G1, cooldown: слотов в unknown {b50.get('unknown_alerts_cooldown', 0):,} из {b50.get('alerts_cooldown', 0):,}; \"\n                  f\"recall [{_f(b50.get('recall_low_cooldown'))}; {_f(b50.get('recall_up_cooldown'))}], \"\n                  f\"precision [{_f(b50.get('precision_cooldown'))}; {_f(b50.get('precision_upper_cooldown'))}].\", \"\"]\n            if \"competing_d1\" in f:\n                cd = f[\"competing_d1\"]\n                L += [f\"Onset в D+1 (competing): {_f(cd['share_of_rows'])} строк, {_f(cd['share_of_positives'])} положительных. \"\n                      \"PR-AUC для «первого onset» (onset в D+2 без onset в D+1): \"\n                      + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in cd[\"pr_auc_first_onset\"].items()), \"\"]\n        d = r[\"decision\"]\n        if r.get(\"stress_2021\", {}).get(\"pr_auc\"):\n            st = r[\"stress_2021\"]\n            L += [f\"Stress 2021 (модели fold_2024, в выборе не участвует): строк {st['rows']:,}, prevalence {_f(st['prevalence'], 4)}; PR-AUC \"\n                  + \", \".join(f\"{k.split('_')[0]} {_f(v)}\" for k, v in st[\"pr_auc\"].items()), \"\"]\n        L += [\"Gates: \" + \", \".join(f\"{k} = {'—' if v is None else ('да' if v else 'нет')}\" for k, v in d[\"gates\"].items()), \"\"]\n    L += [\"\", \"## Как читать\", \"\",\n          \"- `validated_proxy_model` — модель (LightGBM или логистическая регрессия) лучше простых baseline (приор по типу, правило) \"\n          \"в обоих фолдах: нижняя граница 95% CI разницы PR-AUC > 0; при равенстве берётся логистическая регрессия.\",\n          \"- `baseline_only` — модель не доказала преимущества; бэкенд отдаёт скор простого правила/приора.\",\n          \"- `insufficient_events` — мало событий в валидации: бэкенд не выдаёт скор (abstain).\",\n          \"- P@k — precision топ-k строк в сутки среди размеченных строк; `cd` — с cooldown по каналу.\",\n          \"\"]\n    return \"\\n\".join(L)\n\n\n# ---------------------------------------------------------------------------\n# Синтетика для самопроверок и тестов\n# ---------------------------------------------------------------------------\ndef synthetic_panel_v3(out_dir: Path, n_channels: int = 150, seed: int = 3, signal: float = 2.0,\n                       end: date = date(2025, 1, 10), gas_features: bool = False) -> Path:\n    \"\"\"Панель со схемой v3 и встроенным сигналом (для тестов; не реальные данные).\n    gas_features: добавить f_gas_n_7d / f_gas_last и строки above_threshold_at_D для T4 (тесты экспорта 16).\"\"\"\n    import polars as pl\n\n    rng = np.random.default_rng(seed)\n    days = [date(2019, 1, 1) + timedelta(days=i) for i in range(0, (end - date(2019, 1, 1)).days, 3)]\n    n = n_channels * len(days)\n    ch = np.repeat(np.arange(n_channels), len(days))\n    d = np.tile(np.array(days, dtype=\"datetime64[D]\"), n_channels)\n    types = np.array([\"Состояние насоса\", \"КД Дверь\", \"Газовый датчик\", \"Состояние фазы\"])[ch % 4]\n    base = rng.poisson(0.3, n).astype(float)\n    frame = {\n        \"d_channel_key\": np.array([hashlib.sha256(f\"s{c}\".encode()).hexdigest()[:16] for c in range(n_channels)])[ch],\n        \"d_cutoff_date\": d, \"d_label_decision_end\": d + np.timedelta64(4, \"D\"),\n        \"тип_датчика\": types, \"тип_инж_системы\": np.where(ch % 2 == 0, \"ВК\", \"ЭС\"),\n        \"link_state\": rng.choice([\"O\", \"F\", \"A\"], n, p=[0.9, 0.05, 0.05]),\n        \"power_state\": rng.choice([\"O\", \"F\"], n, p=[0.9, 0.1]),\n        \"d_weekday\": (d.astype(\"datetime64[D]\").astype(int) + 3) % 7 + 1,\n        \"d_month\": d.astype(\"datetime64[M]\").astype(int) % 12 + 1,\n        \"f_n_events_7d\": rng.poisson(50, n).astype(float),\n    }\n    for t, spec in TARGET_SPECS.items():\n        for col in spec[\"rule\"]:\n            frame.setdefault(col, base + rng.poisson(0.2, n))\n    noise = rng.normal(0, 1, n)\n    for t, spec in TARGET_SPECS.items():\n        x = frame[spec[\"rule\"][0]]\n        logit = -4.0 + signal * (x - x.mean()) / (x.std() + 1e-9) + 0.3 * noise\n        y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(float)\n        null = rng.random(n) < 0.3\n        frame[t] = np.where(null, np.nan, y)\n        frame[\"reason_\" + t[len(\"target_\"):]] = np.where(null, \"unobserved_window\", np.where(y == 1, \"positive\", \"negative\"))\n    if gas_features:\n        # объект = 8 каналов; ежегодное «молчание» объекта (как ППР): gas_max = null 12 суток в году\n        obj = ch // 8\n        frame[\"d_object_key\"] = np.array([hashlib.sha256(f\"o{o}\".encode()).hexdigest()[:16] for o in range(n_channels // 8 + 1)])[obj]\n        doy = (d.astype(\"datetime64[D]\") - d.astype(\"datetime64[Y]\")).astype(int)\n        silent = np.abs(doy - (40 + 23 * (obj % 12))) < 6\n        frame[\"gas_max\"] = np.where(silent, np.nan, rng.uniform(0, 0.9, n))\n        frame[\"f_gas_cross_n_30d\"] = rng.poisson(0.3, n).astype(float)\n        above = rng.random(n) < 0.1\n        frame[\"f_gas_n_7d\"] = rng.poisson(5, n).astype(float) + 1\n        frame[\"f_gas_last\"] = np.where(above, 1.5, rng.uniform(0, 0.9, n))\n        frame[\"target_t4_gas_cross\"] = np.where(above, np.nan, frame[\"target_t4_gas_cross\"])\n        frame[\"reason_t4_gas_cross\"] = np.where(above, \"above_threshold_at_D\", frame[\"reason_t4_gas_cross\"])\n    panel = pl.DataFrame(frame).with_columns(\n        pl.col(\"d_cutoff_date\").cast(pl.Date), pl.col(\"d_label_decision_end\").cast(pl.Date),\n        *[pl.col(t).fill_nan(None).cast(pl.Int8) for t in TARGET_SPECS])\n    panel_dir = Path(out_dir) / \"event_panel_v3\"\n    panel_dir.mkdir(parents=True, exist_ok=True)\n    part = panel_dir / \"event_panel_v3_part00.parquet\"\n    panel.write_parquet(part)\n    feats = [c for c in panel.columns if c.startswith(\"f_\")] + CATEGORICAL + CALENDAR\n    manifest = {\"schema_version\": SCHEMA_VERSION, \"rows\": panel.height, \"taxonomy_sha256\": \"synthetic\",\n                \"panel_config\": {\"synthetic\": True}, \"feature_columns\": feats, \"target_columns\": list(TARGET_SPECS),\n                \"parts\": [{\"file\": part.name, \"sha256\": file_sha256(part), \"rows\": panel.height}]}\n    (panel_dir / \"panel_manifest_v3.json\").write_text(json.dumps(manifest), encoding=\"utf-8\")\n    return panel_dir\n\n\ndef run_targets_self_tests(work_dir: Path) -> dict[str, bool]:\n    import polars as pl\n\n    work_dir = Path(work_dir)\n    rep: dict[str, bool] = {}\n    y = np.array([0, 1, 0, 1, 0, 0])\n    days = np.array([0, 0, 0, 1, 1, 1])\n    ch = np.array([\"a\", \"b\", \"c\", \"b\", \"a\", \"c\"])\n    s = np.array([0.1, 0.9, 0.2, 0.8, 0.3, 0.1])\n    b = budget_curve(days, ch, y, s, [1], cooldown_days=3)[\"1\"]\n    rep[\"budget_top1_hits_both_days\"] = b[\"precision\"] == 1.0 and b[\"recall\"] == 1.0\n    rep[\"cooldown_blocks_repeat_channel\"] = b[\"alerts_cooldown\"] == 2 and b[\"recall_cooldown\"] == 0.5\n    bu = budget_curve(np.array([0, 0]), np.array([\"a\", \"b\"]), np.array([-1, 1]), np.array([0.9, 0.1]), [1], 3)[\"1\"]\n    rep[\"unknown_slot_counted_as_bounds\"] = (bu[\"precision\"] == 0.0 and bu[\"precision_upper\"] == 1.0 and bu[\"unknown_alerts\"] == 1\n                                             and bu[\"recall_low\"] == 0.0 and bu[\"recall_up\"] == 0.5)\n    try:\n        safe_features({\"feature_columns\": [\"f_x\", \"competing_link_onset_d1\"]}, [\"f_x\", \"competing_link_onset_d1\"])\n        rep[\"future_columns_rejected\"] = False\n    except ValueError:\n        rep[\"future_columns_rejected\"] = True\n    frame = pl.DataFrame({\"d_cutoff_date\": [date(2022, 12, 25), date(2022, 12, 30), date(2023, 6, 29), date(2023, 7, 3)],\n                          \"d_label_decision_end\": [date(2022, 12, 29), date(2023, 1, 3), date(2023, 7, 3), date(2023, 7, 7)]})\n    tr, ca, va = split_fold(pl, frame, FOLDS[0])\n    rep[\"purging_by_label_decision_end\"] = tr.height == 1 and ca.height == 0 and va.height == 1\n    root = work_dir / \"synthetic_in\"\n    synthetic_panel_v3(root, n_channels=60)\n    cfg = make_config_targets(\"SMOKE\", {\"input_dir\": str(root), \"output_dir\": str(work_dir / \"out\"),\n                                        \"smoke_channel_share\": 1, \"targets\": [\"target_t2a_link_onset\"],\n                                        \"bootstrap_reps\": 10})\n    res = run_targets_v3(cfg, log=lambda *a: None)\n    rep[\"synthetic_end_to_end\"] = res[\"status\"] == \"completed_smoke_non_comparable\"\n    text = (work_dir / \"out\" / \"results_targets_v3.json\").read_text(encoding=\"utf-8\")\n    keys = pl.read_parquet(root / \"event_panel_v3\" / \"event_panel_v3_part00.parquet\")[\"d_channel_key\"].unique().to_list()\n    rep[\"no_channel_keys_in_outputs\"] = not any(k in text for k in keys)\n    return rep\n",
"gas_bundle_v3.py": "\"\"\"Ноутбук 16: единственный финальный train G2_no_calendar для target_t4_gas_cross и исполняемый bundle для backend.\n\nРешения зафиксированы до запуска (Drive 22–27, docs/EXPORT_16_GAS_BUNDLE_V3_RU.md):\n- цель и модель не выбираются заново: G2_no_calendar из ноутбука 15 (те же признаки без календаря, гиперпараметры,\n  выборка отрицательных 30:1 с весами, seed); код обучения — функции target_models_v3_runtime без изменений;\n- один рецепт на прогон (RECIPE): refit_2025h1 (train < 2025, Platt на 2025 H1, audit 2025 H2)\n  или fold_2024_exact (точное воспроизведение fold_2024 из 15); audit ничего не выбирает;\n- остановка до обучения, если в калибровке < 30 известных положительных (решение по числу строк, не по метрикам);\n- выход: bundle без pickle (model.txt LightGBM, bundle.json, predictor.py, код признаков 14, контракт, пример,\n  requirements, SHA-256) + audit и acceptance — только агрегаты, без ключей каналов и построчных прогнозов.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nimport zipfile\nfrom datetime import date, datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nimport target_models_v3_runtime as rt\n\nTARGET = \"target_t4_gas_cross\"\nMODEL_NAME = \"G2_no_calendar\"\nBUNDLE_SCHEMA = \"gas_cross_bundle_v3.1\"\nTARGET_CODE = \"gas_threshold_cross_1pct_d2_proxy_v3\"\nRECIPES: dict[str, dict[str, Any]] = {\n    \"refit_2025h1\": {\"train_end\": date(2025, 1, 1), \"calibration\": (date(2025, 1, 1), date(2025, 7, 1)),\n                     \"audit\": (date(2025, 7, 1), date(2026, 1, 1))},\n    \"fold_2024_exact\": {\"train_end\": date(2024, 1, 1), \"calibration\": (date(2024, 1, 1), date(2024, 7, 1)),\n                        \"audit\": (date(2024, 7, 1), date(2025, 1, 1))},\n}\nDEFAULT_RECIPE = \"refit_2025h1\"\nHISTORY_DAYS_REQUIRED = 400\nEXPECTED_15_G2_FOLD_2024_PR_AUC = 0.39233349861671724\nBUNDLE_DIR_NAME = \"gas_cross_v3_bundle\"\n\n\nclass RecipeStop(RuntimeError):\n    \"\"\"Рецепт нельзя обучить по заранее заданному правилу (мало положительных) — решение за человеком.\"\"\"\n\n\nclass ReleaseBlocked(RuntimeError):\n    \"\"\"Release gate не пройден (сверка с 15 или приёмка): принятый bundle не выдаётся.\"\"\"\n\n\ndef make_config_export(mode: str, recipe: str = DEFAULT_RECIPE, overrides: dict[str, Any] | None = None) -> dict[str, Any]:\n    if recipe not in RECIPES:\n        raise ValueError(f\"RECIPE должен быть одним из {list(RECIPES)}\")\n    cfg = rt.make_config_targets(mode)\n    cfg.update({\"recipe\": recipe, \"min_calibration_positives\": 30, \"min_train_positives\": 30,\n                \"acceptance_rows\": 20_000, \"latency_rows\": 100_000, \"latency_budget_s\": 300.0,\n                # PR-AUC G2 fold_2024 из results_15_v3.json (FULL, 2026-09-24): проверка, что код экспорта = код 15\n                \"expected_15_fold_2024_pr_auc\": EXPECTED_15_G2_FOLD_2024_PR_AUC if mode == \"FULL\" else None,\n                \"reproduction_tolerance\": 1e-3,\n                \"code_dir\": os.environ.get(\"LDT_CODE_DIR\", str(Path(__file__).resolve().parent))})\n    cfg.update(overrides or {})\n    return cfg\n\n\ndef _json_default(o: Any) -> Any:\n    if isinstance(o, np.generic):\n        return o.item()\n    if isinstance(o, (date, datetime)):\n        return o.isoformat()\n    raise TypeError(f\"не сериализуется в JSON: {type(o).__name__}\")\n\n\ndef _dump(obj: Any) -> str:\n    return json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default)\n\n\ndef _sha256_file(path: Path) -> str:\n    return rt.file_sha256(Path(path))\n\n\ndef _sha256_text(text: str) -> str:\n    return hashlib.sha256(text.encode(\"utf-8\")).hexdigest()\n\n\ndef g2_features(manifest: dict[str, Any], columns: list[str]) -> list[str]:\n    \"\"\"Тот же список, что у G2 в 15: allowlist manifest + справочник, минус календарь. Число не задаётся вручную.\"\"\"\n    return [c for c in rt.safe_features(manifest, columns) if c not in rt.CALENDAR]\n\n\ndef recipe_fold(name: str) -> dict[str, Any]:\n    r = RECIPES[name]\n    return {\"name\": name, \"train_end\": r[\"train_end\"], \"calibration\": r[\"calibration\"], \"validation\": r[\"audit\"]}\n\n\n# ---------------------------------------------------------------------------\n# Обучение и audit\n# ---------------------------------------------------------------------------\ndef train_final(pl: Any, frame: Any, feats: list[str], cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    fold = recipe_fold(cfg[\"recipe\"])\n    train, cal, audit_all = rt.split_fold(pl, frame, fold)\n    known = pl.col(TARGET).is_not_null()\n    train, cal = train.filter(known), cal.filter(known)\n    counts = {\"train\": train.height, \"train_pos\": int(train[TARGET].sum() or 0),\n              \"calibration\": cal.height, \"calibration_pos\": int(cal[TARGET].sum() or 0),\n              \"audit_selectable\": audit_all.height, \"audit_known\": int(audit_all[TARGET].is_not_null().sum())}\n    log(f\"рецепт {cfg['recipe']}: train {counts['train']:,} (1: {counts['train_pos']:,}), \"\n        f\"калибровка {counts['calibration']:,} (1: {counts['calibration_pos']:,}), audit selectable {counts['audit_selectable']:,}\")\n    if counts[\"calibration_pos\"] < cfg[\"min_calibration_positives\"] or counts[\"train_pos\"] < cfg[\"min_train_positives\"]:\n        raise RecipeStop(f\"рецепт {cfg['recipe']}: положительных в train {counts['train_pos']}, в калибровке \"\n                         f\"{counts['calibration_pos']} (нужно ≥ {cfg['min_train_positives']} / \"\n                         f\"≥ {cfg['min_calibration_positives']}). Обучение не запускалось; решение о рецепте — за Дашей.\")\n    t0 = time.time()\n    tr = rt.sample_train(pl, train, TARGET, cfg)\n    model = rt.LightGBM().fit(tr, TARGET, feats, cfg)\n    raw_cal = model.score(cal)\n    y_cal = cal[TARGET].to_numpy().astype(int)\n    platt = rt.platt(raw_cal, y_cal, True)\n    if platt is None:\n        raise RecipeStop(\"в калибровке один класс — Platt не определён\")\n    counts[\"train_sampled\"] = tr.height\n    log(f\"обучение и калибровка: {time.time() - t0:.0f} с\")\n    return {\"model\": model, \"platt\": platt, \"counts\": counts, \"audit_all\": audit_all,\n            \"prior\": float(train[TARGET].mean()), \"fit_s\": round(time.time() - t0, 1),\n            \"calibration_ece10\": rt.ece(y_cal, rt.apply_platt(platt, raw_cal, True, 0.0)),\n            \"calibration_prevalence\": float(y_cal.mean())}\n\n\ndef runtime_eligible(frame: Any, thr: float) -> np.ndarray:\n    n7 = frame[\"f_gas_n_7d\"].to_numpy().astype(float)\n    last = frame[\"f_gas_last\"].to_numpy().astype(float)\n    return (np.nan_to_num(n7) > 0) & ~np.isnan(last) & (last < thr)\n\n\ndef audit_final(pl: Any, fit: dict[str, Any], cfg: dict[str, Any], thr: float) -> dict[str, Any]:\n    \"\"\"Однократная проверка на отложенном окне рецепта. Ничего не выбирает; R1 — comparator.\"\"\"\n    from sklearn.metrics import average_precision_score, roc_auc_score\n\n    frame = fit[\"audit_all\"]\n    if frame.height == 0:\n        return {\"status\": \"no_audit_rows\"}\n    y_all = frame[TARGET].fill_null(-1).to_numpy().astype(int)\n    known = y_all >= 0\n    days = np.array([(d - date(2019, 1, 1)).days for d in frame[\"d_cutoff_date\"].to_list()])\n    chans = frame[\"d_channel_key\"].to_numpy()\n    raw_g2 = fit[\"model\"].score(frame)\n    raw_r1 = rt.rule_score(frame, rt.TARGET_SPECS[TARGET])\n    elig = runtime_eligible(frame, thr)\n    out: dict[str, Any] = {\"rows_selectable\": int(frame.height), \"rows_known\": int(known.sum()),\n                           \"positives_known\": int((y_all == 1).sum()),\n                           \"unknown_share_selectable\": float(1 - known.mean()),\n                           # все selectable строки должны проходить runtime eligibility (она не смотрит в будущее)\n                           \"runtime_eligible_share_of_selectable\": float(elig.mean()), \"models\": {}}\n    y = y_all[known]\n    ok = np.unique(y).size == 2\n    for name, raw in ((MODEL_NAME, raw_g2), (\"R1_rule\", raw_r1)):\n        budget = rt.budget_curve(days, chans, y_all, raw, cfg[\"budgets_per_day\"], cfg[\"cooldown_days\"])\n        for b in budget.values():\n            b[\"alert_unknown_share\"] = b[\"unknown_alerts\"] / max(b[\"alerts\"], 1)\n            b[\"alert_unknown_share_cooldown\"] = b[\"unknown_alerts_cooldown\"] / max(b[\"alerts_cooldown\"], 1)\n        m = {\"pr_auc\": float(average_precision_score(y, raw[known])) if ok else None,\n             \"roc_auc\": float(roc_auc_score(y, raw[known])) if ok else None, \"budget\": budget}\n        if name == MODEL_NAME:\n            p = rt.apply_platt(fit[\"platt\"], raw[known], True, fit[\"prior\"])\n            m[\"brier\"] = float(np.mean((p - y) ** 2))\n            m[\"ece10\"] = rt.ece(y, p)\n        out[\"models\"][name] = m\n    out[\"prevalence_known\"] = float(y.mean()) if len(y) else None\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Экспорт bundle\n# ---------------------------------------------------------------------------\ndef _feature_spec(model: Any, frame: Any) -> list[dict[str, Any]]:\n    spec = []\n    for c in model.features:\n        if c in rt.CATEGORICAL:\n            spec.append({\"name\": c, \"kind\": \"categorical\", \"dtype\": \"string|null\", \"levels\": model.levels[c]})\n        else:\n            spec.append({\"name\": c, \"kind\": \"numeric\", \"dtype\": \"float32|null\"})\n    return spec\n\n\ndef _example_records(features: list[dict[str, Any]], as_of: str) -> list[dict[str, Any]]:\n    \"\"\"Синтетический пример (не реальные данные): одна подходящая строка, одна с превышенным порогом, одна не газовая.\"\"\"\n    base: dict[str, Any] = {\"as_of_date\": as_of, \"ид_канала_данных\": \"EXAMPLE-CHANNEL-1\", \"request_id\": \"ex-1\"}\n    for f in features:\n        base[f[\"name\"]] = (f[\"levels\"][0] if f[\"levels\"] else None) if f[\"kind\"] == \"categorical\" else 0.0\n    base.update({\"f_gas_n_7d\": 120.0, \"f_gas_last\": 0.05, \"f_gas_max_7d\": 0.4})\n    above = {**base, \"ид_канала_данных\": \"EXAMPLE-CHANNEL-2\", \"request_id\": \"ex-2\", \"f_gas_last\": 1.3}\n    not_gas = {**base, \"ид_канала_данных\": \"EXAMPLE-CHANNEL-3\", \"request_id\": \"ex-3\", \"f_gas_n_7d\": 0.0,\n               \"f_gas_last\": None, \"f_gas_max_7d\": None}\n    return [base, above, not_gas]\n\n\ndef _versions() -> dict[str, str]:\n    import lightgbm\n    import pandas\n    import polars\n    import sklearn\n\n    return {\"numpy\": np.__version__, \"pandas\": pandas.__version__, \"lightgbm\": lightgbm.__version__,\n            \"polars\": polars.__version__, \"scikit-learn(train only)\": sklearn.__version__}\n\n\ndef _model_version(recipe: str, model_sha: str) -> str:\n    return f\"gas-cross-g2-v3-{recipe}-{model_sha[:12]}\"\n\n\ndef _feature_contract_version(schema: Any, taxonomy_sha: Any, allowlist_sha: str) -> str:\n    return f\"event-panel-v{schema}+tax-{str(taxonomy_sha)[:8]}+g2-{allowlist_sha[:8]}\"\n\n\ndef _contract(spec: dict[str, Any], features: list[dict[str, Any]]) -> dict[str, Any]:\n    return {\"bundle_schema\": BUNDLE_SCHEMA, \"target_code\": TARGET_CODE,\n                \"input\": {\"required\": [\"as_of_date (YYYY-MM-DD) — обязательна, иначе ContractError\"],\n                          \"passthrough\": [\"ид_канала_данных\", \"request_id\"],\n                          \"features\": features, \"n_features\": len(features),\n                          \"unknown_fields\": \"ошибка контракта\", \"missing_features\": \"ошибка контракта\",\n                          \"producer\": \"predictor.build_features(journal_files, catalogue_file, as_of_date, bundle) — код ноутбука 14\",\n                          \"history_days_required\": spec.get(\"history_days_required\", HISTORY_DAYS_REQUIRED)},\n                \"model_version\": spec[\"model_version\"], \"feature_contract_version\": spec[\"feature_contract_version\"],\n                \"output_fields\": [\"request_index\", \"as_of_date\", \"observation_cutoff_date\", \"ид_канала_данных\", \"request_id\",\n                                  \"target_code\", \"model_version\", \"feature_contract_version\", \"evidence_level\",\n                                  \"incident_type\", \"score\", \"raw_score\", \"score_kind\", \"decision_status\", \"reason_codes\",\n                                  \"window_start\", \"window_end_exclusive\", \"evidence_scope\", \"operational_ready\",\n                                  \"not_a_confirmed_incident\", \"not_fire_probability\"],\n                \"eligibility\": spec[\"eligibility\"]}\n\n\ndef export_bundle(out_dir: Path, fit: dict[str, Any], manifest: dict[str, Any], manifest_path: Path,\n                  cfg: dict[str, Any], audit: dict[str, Any], thr: float) -> Path:\n    code_dir = Path(cfg[\"code_dir\"])\n    bdir = Path(out_dir) / BUNDLE_DIR_NAME\n    if bdir.exists():\n        shutil.rmtree(bdir)\n    (bdir / \"features_v3\").mkdir(parents=True)\n    model = fit[\"model\"]\n    booster = model.model.booster_\n    model_txt = booster.model_to_string()\n    (bdir / \"model.txt\").write_text(model_txt, encoding=\"utf-8\")\n    features = _feature_spec(model, fit[\"audit_all\"])\n    names = [f[\"name\"] for f in features]\n    if booster.feature_name() != names or booster.num_feature() != len(names):\n        raise RuntimeError(\"признаки booster не совпадают с экспортируемым контрактом\")\n    allowlist = list(manifest[\"feature_columns\"])\n    recipe = RECIPES[cfg[\"recipe\"]]\n    iso = lambda d: d.isoformat()  # noqa: E731\n    spec = {\n        \"bundle_schema\": BUNDLE_SCHEMA, \"created_at_utc\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\"),\n        \"mode\": cfg[\"mode\"], \"target\": TARGET, \"target_code\": TARGET_CODE, \"incident_type\": \"gas_threshold_cross\",\n        \"model\": MODEL_NAME, \"model_format\": \"lightgbm_text\", \"model_sha256\": _sha256_text(model_txt),\n        \"score_kind\": \"proxy_score\", \"decision_status\": \"experimental_shadow\",\n        # E1 — наблюдаемый в журнале proxy одного канала (шкала scorecard 12: E1/E2 proxy, E4 синтетика)\n        \"evidence_level\": \"E1\", \"evidence_level_meaning\": \"observed_journal_proxy_not_confirmed_incident\",\n        \"model_version\": _model_version(cfg[\"recipe\"], _sha256_text(model_txt)),\n        \"feature_contract_version\": _feature_contract_version(\n            manifest.get(\"schema_version\"), manifest.get(\"taxonomy_sha256\"),\n            _sha256_text(json.dumps(list(manifest[\"feature_columns\"]), ensure_ascii=False))),\n        \"evidence_scope\": \"known_outcome_subset\", \"operational_ready\": False,\n        \"meaning\": (\"score будущего наблюдаемого строгого пересечения газового порога 1 % снизу вверх в [D+2; D+3) \"\n                    \"по данным до конца D; не вероятность пожара и не подтверждённый инцидент\"),\n        \"window\": {\"start_offset_days\": 2, \"end_offset_days_exclusive\": 3, \"features_until\": \"end of day D\"},\n        \"recipe\": {\"name\": cfg[\"recipe\"], \"train\": f\"D < {iso(recipe['train_end'])}, d_label_decision_end ≤ {iso(recipe['calibration'][0])}\",\n                   \"calibration\": f\"[{iso(recipe['calibration'][0])}; {iso(recipe['calibration'][1])}), decision_end ≤ {iso(recipe['audit'][0])}\",\n                   \"audit\": f\"[{iso(recipe['audit'][0])}; {iso(recipe['audit'][1])}), decision_end ≤ {iso(recipe['audit'][1])}\",\n                   \"excluded\": f\"[{iso(rt.EXCLUDED[0])}; {iso(rt.EXCLUDED[1])})\",\n                   \"hyperparameters\": {k: v for k, v in model.model.get_params().items()\n                                       if k in (\"n_estimators\", \"learning_rate\", \"num_leaves\", \"max_bin\", \"min_child_samples\",\n                                                \"subsample\", \"subsample_freq\", \"colsample_bytree\", \"random_state\")},\n                   \"negatives_per_positive\": cfg[\"max_neg_per_pos\"], \"max_train_rows\": cfg[\"max_train_rows\"]},\n        \"counts\": fit[\"counts\"],\n        \"platt\": {\"coef\": float(fit[\"platt\"].coef_[0][0]), \"intercept\": float(fit[\"platt\"].intercept_[0]),\n                  \"input\": \"logit(clip(raw, 1e-6, 1-1e-6))\"},\n        \"features\": features, \"n_features\": len(features),\n        \"feature_allowlist_sha256\": _sha256_text(json.dumps(allowlist, ensure_ascii=False)),\n        \"eligibility\": {\"rule\": \"f_gas_n_7d > 0 and f_gas_last is not null and f_gas_last < gas_threshold_percent\",\n                        \"gas_threshold_percent\": thr,\n                        \"abstain_reasons\": [\"NOT_GAS_STREAM\", \"NO_GAS_READING_AT_D\", \"ABOVE_THRESHOLD_AT_D\"]},\n        \"history_days_required\": HISTORY_DAYS_REQUIRED,\n        \"panel_config\": manifest.get(\"panel_config\", {}),\n        \"panel\": {\"schema_version\": manifest.get(\"schema_version\"), \"rows\": manifest.get(\"rows\"),\n                  \"manifest_sha256\": _sha256_file(manifest_path), \"taxonomy_sha256\": manifest.get(\"taxonomy_sha256\"),\n                  \"data_end\": manifest.get(\"data_end\"), \"parts_sha256\": [p[\"sha256\"] for p in manifest.get(\"parts\", [])]},\n        \"versions\": _versions(),\n        \"backend_rules\": [\"автоматические заявки/критические уведомления по score выключены (shadow)\",\n                          \"суточный бюджет top-k и cooldown 72 ч применяет backend; рекомендуемый shadow-бюджет 25/сутки\",\n                          \"score разных целей не усредняются и не называются вероятностью инцидента\",\n                          \"физической локации нет: показывать объект/канал, координаты — отсутствуют\"],\n    }\n    (bdir / \"bundle.json\").write_text(_dump(spec), encoding=\"utf-8\")\n    shutil.copy(code_dir / \"gas_predictor_v3.py\", bdir / \"predictor.py\")\n    shutil.copy(code_dir / \"event_panel_v3.py\", bdir / \"features_v3\" / \"event_panel_v3.py\")\n    shutil.copy(code_dir / \"config\" / \"state_taxonomy_v3.json\", bdir / \"features_v3\" / \"state_taxonomy_v3.json\")\n    contract = _contract(spec, features)\n    (bdir / \"feature_contract.json\").write_text(_dump(contract), encoding=\"utf-8\")\n    example = _example_records(features, \"2026-01-15\")\n    (bdir / \"example_request.json\").write_text(_dump(example), encoding=\"utf-8\")\n    v = spec[\"versions\"]\n    (bdir / \"requirements.txt\").write_text(\n        f\"numpy=={v['numpy']}\\npandas=={v['pandas']}\\nlightgbm=={v['lightgbm']}\\n\"\n        f\"# только для build_features (признаки из журнала):\\npolars=={v['polars']}\\n\", encoding=\"utf-8\")\n    (bdir / \"audit.json\").write_text(_dump(audit), encoding=\"utf-8\")\n    (bdir / \"README_RU.md\").write_text(bundle_readme(spec, audit), encoding=\"utf-8\")\n    _pack(bdir, Path(out_dir))\n    return bdir\n\n\ndef _pack(bdir: Path, out_dir: Path) -> Path:\n    \"\"\"SHA256SUMS.json по всем файлам каталога и ZIP (SHA-256 самого ZIP — отдельным файлом, не внутри).\"\"\"\n    sums = {str(p.relative_to(bdir)): _sha256_file(p) for p in sorted(bdir.rglob(\"*\")) if p.is_file()}\n    (bdir / \"SHA256SUMS.json\").write_text(_dump(sums), encoding=\"utf-8\")\n    zpath = out_dir / f\"{BUNDLE_DIR_NAME}.zip\"\n    with zipfile.ZipFile(zpath, \"w\", zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(bdir.rglob(\"*\")):\n            if p.is_file():\n                z.write(p, Path(BUNDLE_DIR_NAME) / p.relative_to(bdir))\n    return zpath\n\n\ndef bundle_readme(spec: dict[str, Any], audit: dict[str, Any]) -> str:\n    g = audit.get(\"models\", {}).get(MODEL_NAME, {})\n    r = audit.get(\"models\", {}).get(\"R1_rule\", {})\n    b25 = g.get(\"budget\", {}).get(\"25\", {})\n    return \"\\n\".join([\n        f\"# Газовый proxy v3 — bundle для backend ({spec['decision_status']})\", \"\",\n        spec[\"meaning\"] + \".\", \"\",\n        \"```python\",\n        \"from predictor import load_bundle, build_features, predict\",\n        \"bundle = load_bundle('.')\",\n        \"records = build_features(['journal_extract.csv'], 'catalogue.csv', '2026-09-24', bundle)\",\n        \"results = predict(records, bundle)\",\n        \"```\", \"\",\n        f\"- Рецепт: `{spec['recipe']['name']}`; train {spec['recipe']['train']}; калибровка {spec['recipe']['calibration']}.\",\n        f\"- Признаков: {spec['n_features']} (список и типы — `feature_contract.json`); история журнала ≥ \"\n        f\"{spec['history_days_required']} суток до D.\",\n        f\"- Eligibility: {spec['eligibility']['rule']}; иначе `abstain` с причиной.\",\n        f\"- Audit ({spec['recipe']['audit']}): PR-AUC G2 {_fmt(g.get('pr_auc'))} против правила {_fmt(r.get('pr_auc'))}; \"\n        f\"25/сутки с cooldown: precision {_fmt(b25.get('precision_cooldown'))}–{_fmt(b25.get('precision_upper_cooldown'))}, \"\n        f\"доля неизвестных исходов среди оповещений {_fmt(b25.get('alert_unknown_share_cooldown'))}.\",\n        \"- `operational_ready = false`: автоматические заявки по score выключены; бюджет и cooldown — на backend.\",\n        \"- Файлы: `model.txt` (LightGBM), `bundle.json` (калибровка, контракт), `predictor.py`, `features_v3/` (код \"\n        \"признаков ноутбука 14 и таксономия), `SHA256SUMS.json`.\", \"\"])\n\n\ndef _fmt(x: Any, d: int = 3) -> str:\n    return \"—\" if x is None else f\"{x:.{d}f}\"\n\n\n# ---------------------------------------------------------------------------\n# Приёмка: bundle, загруженный из файлов, против модели в памяти (только агрегаты)\n# ---------------------------------------------------------------------------\ndef _import_predictor(bundle_dir: Path) -> Any:\n    import importlib.util\n    import sys\n\n    spec = importlib.util.spec_from_file_location(\"gas_bundle_predictor\", Path(bundle_dir) / \"predictor.py\")\n    module = importlib.util.module_from_spec(spec)\n    flag, sys.dont_write_bytecode = sys.dont_write_bytecode, True   # не оставлять __pycache__ в каталоге bundle\n    try:\n        spec.loader.exec_module(module)\n    finally:\n        sys.dont_write_bytecode = flag\n    return module\n\n\ndef acceptance(bundle_dir: Path, fit: dict[str, Any], cfg: dict[str, Any], thr: float) -> dict[str, Any]:\n    import tempfile\n\n    import polars as pl\n\n    pred = _import_predictor(bundle_dir)\n    bundle = pred.load_bundle(bundle_dir)\n    names = [f[\"name\"] for f in bundle[\"spec\"][\"features\"]]\n    frame = fit[\"audit_all\"]\n    if frame.height > cfg[\"acceptance_rows\"]:\n        frame = frame.sample(n=cfg[\"acceptance_rows\"], seed=cfg[\"random_seed\"])\n    rep: dict[str, Any] = {\"rows_checked\": frame.height}\n    elig = runtime_eligible(frame, thr)\n    sub = frame.filter(elig)\n    records = sub.select([pl.col(\"d_cutoff_date\").cast(pl.String).alias(\"as_of_date\")] + names).to_pandas()\n    res = pred.predict(records, bundle)\n    raw_mem = fit[\"model\"].score(sub)\n    cal_mem = rt.apply_platt(fit[\"platt\"], raw_mem, True, fit[\"prior\"])\n    raw_b = np.array([r[\"raw_score\"] for r in res], dtype=float)\n    cal_b = np.array([r[\"score\"] for r in res], dtype=float)\n    rep[\"max_abs_diff_raw\"] = float(np.max(np.abs(raw_b - raw_mem))) if len(res) else 0.0\n    rep[\"max_abs_diff_calibrated\"] = float(np.max(np.abs(cal_b - cal_mem))) if len(res) else 0.0\n    rep[\"parity_ok\"] = rep[\"max_abs_diff_raw\"] < 1e-9 and rep[\"max_abs_diff_calibrated\"] < 1e-9\n    rep[\"all_selectable_rows_runtime_eligible\"] = bool(elig.all())\n    ex = json.loads((Path(bundle_dir) / \"example_request.json\").read_text(encoding=\"utf-8\"))\n    out = pred.predict(ex, bundle)\n    rep[\"example_statuses\"] = [o[\"decision_status\"] + \":\" + \",\".join(o[\"reason_codes\"]) for o in out]\n    rep[\"example_ok\"] = (out[0][\"decision_status\"] == \"experimental_shadow\" and out[0][\"score\"] is not None\n                         and out[1][\"reason_codes\"] == [\"ABOVE_THRESHOLD_AT_D\"] and out[2][\"reason_codes\"] == [\"NOT_GAS_STREAM\"])\n    bad = dict(ex[0])\n    bad.pop(names[0])\n    try:\n        pred.predict([bad], bundle)\n        rep[\"missing_feature_rejected\"] = False\n    except pred.ContractError:\n        rep[\"missing_feature_rejected\"] = True\n    try:\n        pred.predict([{**ex[0], \"target_t4_gas_cross\": 1}], bundle)\n        rep[\"unknown_field_rejected\"] = False\n    except pred.ContractError:\n        rep[\"unknown_field_rejected\"] = True\n    no_date = {k: v for k, v in ex[0].items() if k != \"as_of_date\"}\n    for key, rec in ((\"missing_as_of_rejected\", no_date), (\"bad_as_of_rejected\", {**ex[0], \"as_of_date\": \"25.09.2026\"})):\n        try:\n            pred.predict([rec], bundle)\n            rep[key] = False\n        except pred.ContractError:\n            rep[key] = True\n    with tempfile.TemporaryDirectory() as tmp:\n        copy = Path(tmp) / \"b\"\n        shutil.copytree(bundle_dir, copy)\n        with open(copy / \"features_v3\" / \"state_taxonomy_v3.json\", \"a\", encoding=\"utf-8\") as f:\n            f.write(\" \")\n        try:\n            pred.load_bundle(copy)\n            rep[\"tampered_file_rejected\"] = False\n        except pred.ContractError:\n            rep[\"tampered_file_rejected\"] = True\n    o = out[0]\n    rep[\"typed_fields_present\"] = all(o.get(k) for k in (\"model_version\", \"feature_contract_version\", \"evidence_level\",\n                                                           \"observation_cutoff_date\", \"window_start\"))\n    cat = next((f[\"name\"] for f in bundle[\"spec\"][\"features\"] if f[\"kind\"] == \"categorical\"), None)\n    if cat:\n        o = pred.predict([{**ex[0], cat: \"никогда-не-встречавшийся-уровень\"}], bundle)[0]\n        rep[\"unknown_category_scored\"] = o[\"score\"] is not None\n    nulls = {**ex[0], **{n: None for n in names if n not in (\"f_gas_n_7d\", \"f_gas_last\")}}\n    rep[\"null_features_scored\"] = pred.predict([nulls], bundle)[0][\"score\"] is not None\n    big = records.sample(n=cfg[\"latency_rows\"], replace=True, random_state=0) if len(records) else records\n    t0 = time.time()\n    if len(big):\n        pred.predict(big, bundle)\n    rep[\"latency_rows\"] = int(len(big))\n    rep[\"latency_s\"] = round(time.time() - t0, 2)\n    rep[\"latency_ok\"] = rep[\"latency_s\"] < cfg[\"latency_budget_s\"]\n    rep[\"passed\"] = all(v for k, v in rep.items() if isinstance(v, bool))\n    return rep\n\n\n# ---------------------------------------------------------------------------\n# Воспроизведение 15 (не участвует в выборе: рецепт и модель уже зафиксированы)\n# ---------------------------------------------------------------------------\ndef reproduce_15(pl: Any, frame: Any, feats: list[str], cfg: dict[str, Any], audit: dict[str, Any]) -> dict[str, Any]:\n    \"\"\"PR-AUC G2 на валидации fold_2024 тем же кодом, что в экспорте, против числа из results_15_v3.json.\n    Для fold_2024_exact это сам audit; для refit — отдельное обучение fold_2024 только ради сверки.\"\"\"\n    from sklearn.metrics import average_precision_score\n\n    expected = cfg.get(\"expected_15_fold_2024_pr_auc\")\n    if cfg[\"recipe\"] == \"fold_2024_exact\":\n        got = audit.get(\"models\", {}).get(MODEL_NAME, {}).get(\"pr_auc\")\n    else:\n        fold = next(f for f in rt.FOLDS if f[\"name\"] == \"fold_2024\")\n        train, _, val = rt.split_fold(pl, frame, fold)\n        known = pl.col(TARGET).is_not_null()\n        train, val = train.filter(known), val.filter(known)\n        if train.height == 0 or val.height == 0 or val[TARGET].n_unique() < 2:\n            return {\"expected\": expected, \"got\": None, \"status\": \"not_applicable\"}\n        m = rt.LightGBM().fit(rt.sample_train(pl, train, TARGET, cfg), TARGET, feats, cfg)\n        got = float(average_precision_score(val[TARGET].to_numpy().astype(int), m.score(val)))\n    rep = {\"expected\": expected, \"got\": got}\n    if expected is None or got is None:\n        rep[\"status\"] = \"not_checked\"\n    else:\n        rep[\"abs_diff\"] = abs(got - expected)\n        rep[\"status\"] = \"reproduced\" if rep[\"abs_diff\"] <= cfg[\"reproduction_tolerance\"] else \"MISMATCH\"\n    return rep\n\n\n# ---------------------------------------------------------------------------\n# Прогон\n# ---------------------------------------------------------------------------\ndef run_export_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:\n    import polars as pl\n\n    t_start = time.time()\n    out = Path(cfg[\"output_dir\"])\n    out.mkdir(parents=True, exist_ok=True)\n    parts, manifest = rt.discover_panel(Path(cfg[\"input_dir\"]), cfg[\"verify_sha256\"])\n    manifest_path = sorted(Path(cfg[\"input_dir\"]).rglob(\"panel_manifest_v3.json\"))[0]\n    thr = float(manifest.get(\"panel_config\", {}).get(\"gas_threshold_percent\", 1.0))\n    columns = pl.scan_parquet(parts[0]).collect_schema().names()\n    feats = g2_features(manifest, columns)\n    log(f\"панель: {manifest['rows']:,} строк, частей {len(parts)}; признаков G2: {len(feats)}; порог газа {thr}\")\n    recipe = RECIPES[cfg[\"recipe\"]]\n    frame = rt.load_target(pl, parts, TARGET, feats, cfg, load_end=recipe[\"audit\"][1])\n    log(f\"selectable строк цели до {recipe['audit'][1]}: {frame.height:,}\")\n    payload: dict[str, Any] = {\"recipe\": cfg[\"recipe\"], \"target\": TARGET, \"model\": MODEL_NAME, \"n_features\": len(feats)}\n\n    def finish(status: str) -> dict[str, Any]:\n        payload.update({\"status\": status, \"runtime_s\": round(time.time() - t_start, 1)})\n        (out / \"results_export_16_v3.json\").write_text(_dump(payload), encoding=\"utf-8\")\n        (out / \"summary_export_16_v3_ru.md\").write_text(summary_markdown(payload), encoding=\"utf-8\")\n        return payload\n\n    def check_repro(repro: dict[str, Any]) -> None:\n        payload[\"reproduction_15\"] = repro\n        log(f\"сверка с 15 (G2 fold_2024 PR-AUC): ожидалось {repro.get('expected')}, получено {repro.get('got')} — {repro['status']}\")\n        if repro[\"status\"] == \"MISMATCH\":   # fail-closed (GPT 29): bundle не собирается, рецепт не меняется\n            finish(\"reproduction_mismatch\")\n            raise ReleaseBlocked(f\"код обучения не воспроизводит G2 из 15: {repro}. Bundle не создан.\")\n\n    if cfg[\"recipe\"] != \"fold_2024_exact\":\n        # сверочное обучение fold_2024 (второй fit, в выборе не участвует) — до финального, чтобы остановиться рано\n        check_repro(reproduce_15(pl, frame, feats, cfg, {}))\n    fit = train_final(pl, frame, feats, cfg, log)\n    audit = audit_final(pl, fit, cfg, thr)\n    payload.update({\"counts\": fit[\"counts\"], \"calibration_ece10\": fit[\"calibration_ece10\"],\n                    \"calibration_prevalence\": fit[\"calibration_prevalence\"], \"audit\": audit})\n    if cfg[\"recipe\"] == \"fold_2024_exact\":\n        check_repro(reproduce_15(pl, frame, feats, cfg, audit))\n    bdir = export_bundle(out, fit, manifest, manifest_path, cfg, audit, thr)\n    acc = acceptance(bdir, fit, cfg, thr)\n    payload[\"acceptance\"] = acc\n    zpath = out / f\"{BUNDLE_DIR_NAME}.zip\"\n    if not acc[\"passed\"]:                   # fail-closed: непринятый bundle не остаётся в Output\n        shutil.rmtree(bdir, ignore_errors=True)\n        zpath.unlink(missing_ok=True)\n        finish(\"acceptance_failed\")\n        raise ReleaseBlocked(f\"приёмка bundle не пройдена: {acc}. Bundle удалён.\")\n    zsha = _sha256_file(zpath)\n    (out / f\"{BUNDLE_DIR_NAME}.zip.sha256\").write_text(f\"{zsha}  {zpath.name}\\n\", encoding=\"utf-8\")\n    payload.update({\"bundle_zip\": zpath.name, \"bundle_zip_sha256\": zsha,\n                    \"model_version\": json.loads((bdir / \"bundle.json\").read_text(encoding=\"utf-8\"))[\"model_version\"]})\n    return finish(\"completed_smoke_non_comparable\" if cfg[\"mode\"] == \"SMOKE\" else \"completed\")\n\n\ndef summary_markdown(p: dict[str, Any]) -> str:\n    if \"counts\" not in p:\n        rp = p.get(\"reproduction_15\", {})\n        return (f\"# 16 · Финальный газовый bundle v3 — {p['status']}\\n\\nОстановлено до финального обучения: сверка с 15 \"\n                f\"(G2 fold_2024 PR-AUC) ожидалось {_fmt(rp.get('expected'), 4)}, получено {_fmt(rp.get('got'), 4)}. \"\n                \"Bundle не создан; рецепт автоматически не меняется.\\n\")\n    a = p[\"audit\"]\n    g = a.get(\"models\", {}).get(MODEL_NAME, {})\n    r = a.get(\"models\", {}).get(\"R1_rule\", {})\n    c = p[\"counts\"]\n    L = [f\"# 16 · Финальный газовый bundle v3 — {p['status']}\", \"\",\n         f\"Рецепт **{p['recipe']}**, модель {MODEL_NAME}, признаков {p['n_features']}. \"\n         \"Цель — наблюдаемое пересечение газа 1 % в D+2 (proxy), не вероятность пожара.\", \"\",\n         f\"- train: {c['train']:,} строк (1: {c['train_pos']:,}), в обучение после выборки {c.get('train_sampled', 0):,};\",\n         f\"- калибровка: {c['calibration']:,} (1: {c['calibration_pos']:,}), ECE-10 на ней {p['calibration_ece10']:.4f};\",\n         f\"- audit: selectable {a.get('rows_selectable', 0):,}, известен исход {a.get('rows_known', 0):,} \"\n         f\"(доля неизвестных {_fmt(a.get('unknown_share_selectable'))}), положительных {a.get('positives_known', 0):,};\",\n         f\"- runtime eligibility покрывает selectable строки audit: {_fmt(a.get('runtime_eligible_share_of_selectable'), 4)} (ожидается 1).\", \"\",\n         \"## Audit (однократно, ничего не выбирает)\", \"\",\n         f\"PR-AUC G2 {_fmt(g.get('pr_auc'))} против правила R1 {_fmt(r.get('pr_auc'))}; ROC-AUC {_fmt(g.get('roc_auc'))}; \"\n         f\"Brier {_fmt(g.get('brier'), 4)}, ECE-10 {_fmt(g.get('ece10'), 4)}.\", \"\",\n         \"| k/сутки | модель | precision (cooldown), границы | recall (cooldown), границы | оповещений | доля unknown среди них |\",\n         \"|---|---|---|---|---|---|\"]\n    for k in (str(x) for x in (5, 10, 25, 50)):\n        for name, m in ((MODEL_NAME, g), (\"R1_rule\", r)):\n            b = m.get(\"budget\", {}).get(k)\n            if b:\n                L.append(f\"| {k} | {name} | {_fmt(b['precision_cooldown'])}–{_fmt(b['precision_upper_cooldown'])} | \"\n                         f\"{_fmt(b['recall_low_cooldown'])}–{_fmt(b['recall_up_cooldown'])} | {b['alerts_cooldown']:,} | \"\n                         f\"{_fmt(b['alert_unknown_share_cooldown'])} |\")\n    acc = p.get(\"acceptance\")\n    rp = p.get(\"reproduction_15\", {})\n    L += [\"\", \"## Сверка с ноутбуком 15\", \"\",\n          f\"PR-AUC G2 на валидации fold_2024: в 15 {_fmt(rp.get('expected'), 4)}, код экспорта {_fmt(rp.get('got'), 4)} — \"\n          f\"**{rp.get('status')}** (в выборе не участвует; MISMATCH — bundle не создаётся).\"]\n    if acc is None:\n        L += [\"\", f\"Bundle не создан: статус {p['status']}.\", \"\"]\n        return \"\\n\".join(L)\n    L += [\"\", \"## Приёмка bundle\", \"\",\n          f\"- parity (bundle из файлов против модели в памяти): raw {acc['max_abs_diff_raw']:.2e}, \"\n          f\"калиброванный {acc['max_abs_diff_calibrated']:.2e} — {'OK' if acc['parity_ok'] else 'FAIL'};\",\n          f\"- пример: {', '.join(acc['example_statuses'])} — {'OK' if acc['example_ok'] else 'FAIL'};\",\n          f\"- контракт: нет признака → ошибка {acc['missing_feature_rejected']}, лишнее поле → ошибка {acc['unknown_field_rejected']}, \"\n          f\"неизвестная категория → score {acc.get('unknown_category_scored')}, null-признаки → score {acc['null_features_scored']};\",\n          f\"- latency: {acc['latency_rows']:,} строк за {acc['latency_s']} с — {'OK' if acc['latency_ok'] else 'FAIL'}.\", \"\",\n          f\"- проверка SHA всех файлов при загрузке, отказ без as_of_date / с битой датой: \"\n          f\"{acc.get('tampered_file_rejected')} / {acc.get('missing_as_of_rejected')} / {acc.get('bad_as_of_rejected')}.\", \"\",\n          \"Ключей каналов и построчных прогнозов в этих файлах нет. Bundle: `\" + p.get(\"bundle_zip\", \"—\") + \"`, \"\n          f\"SHA-256 ZIP `{p.get('bundle_zip_sha256', '—')}` (в `{BUNDLE_DIR_NAME}.zip.sha256`), \"\n          f\"model_version `{p.get('model_version', '—')}`.\", \"\"]\n    return \"\\n\".join(L)\n\n\n# ---------------------------------------------------------------------------\n# Самопроверки на синтетике (запускаются в ноутбуке до прогона)\n# ---------------------------------------------------------------------------\ndef run_export_self_tests(work_dir: Path) -> dict[str, bool]:\n    work_dir = Path(work_dir)\n    rep: dict[str, bool] = {}\n    root = work_dir / \"synthetic_in\"\n    rt.synthetic_panel_v3(root, n_channels=80, end=date(2026, 1, 10), gas_features=True)\n    for recipe in RECIPES:\n        cfg = make_config_export(\"SMOKE\", recipe, {\"input_dir\": str(root), \"output_dir\": str(work_dir / recipe),\n                                                   \"smoke_channel_share\": 1, \"latency_rows\": 2000,\n                                                   \"min_calibration_positives\": 5, \"min_train_positives\": 5})\n        res = run_export_v3(cfg, log=lambda *a: None)\n        rep[f\"{recipe}_acceptance_passed\"] = res[\"acceptance\"][\"passed\"]\n        text = (work_dir / recipe / \"results_export_16_v3.json\").read_text(encoding=\"utf-8\")\n        import polars as pl\n\n        keys = pl.read_parquet(root / \"event_panel_v3\" / \"event_panel_v3_part00.parquet\")[\"d_channel_key\"].unique().to_list()\n        rep[f\"{recipe}_no_channel_keys_in_results\"] = not any(k in text for k in keys)\n    cfg = make_config_export(\"SMOKE\", \"refit_2025h1\", {\"input_dir\": str(root), \"output_dir\": str(work_dir / \"stop\"),\n                                                       \"smoke_channel_share\": 1, \"min_calibration_positives\": 10**9})\n    try:\n        run_export_v3(cfg, log=lambda *a: None)\n        rep[\"too_few_calibration_positives_stops_before_training\"] = False\n    except RecipeStop:\n        rep[\"too_few_calibration_positives_stops_before_training\"] = not (work_dir / \"stop\" / BUNDLE_DIR_NAME).exists()\n    return rep\n\n\n# ---------------------------------------------------------------------------\n# Переупаковка bundle первой версии 16 (без release gates) — без обучения\n# ---------------------------------------------------------------------------\ndef repack_bundle(old_dir: Path, out_dir: Path, code_dir: Path | None = None) -> Path:\n    \"\"\"Тот же model.txt, Platt, признаки и audit; новый predictor.py (обязательная дата, SHA всех файлов, история),\n    typed-поля (model_version, feature_contract_version, evidence_level, observation_cutoff_date), новые\n    SHA256SUMS, ZIP и .sha256. Обучения нет: model_sha256 не меняется.\"\"\"\n    old_dir, out_dir = Path(old_dir), Path(out_dir)\n    code_dir = Path(code_dir) if code_dir else Path(__file__).resolve().parent\n    sums = json.loads((old_dir / \"SHA256SUMS.json\").read_text(encoding=\"utf-8\"))\n    bad = [n for n, sha in sums.items() if _sha256_file(old_dir / n) != sha]\n    if bad:\n        raise RuntimeError(f\"исходный bundle повреждён: {bad}\")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    bdir = out_dir / BUNDLE_DIR_NAME\n    if bdir.exists():\n        shutil.rmtree(bdir)\n    shutil.copytree(old_dir, bdir, ignore=shutil.ignore_patterns(\"__pycache__\"))\n    spec = json.loads((bdir / \"bundle.json\").read_text(encoding=\"utf-8\"))\n    if _sha256_file(bdir / \"model.txt\") != spec[\"model_sha256\"]:\n        raise RuntimeError(\"model.txt не совпадает с bundle.json\")\n    spec.update({\n        \"evidence_level\": \"E1\", \"evidence_level_meaning\": \"observed_journal_proxy_not_confirmed_incident\",\n        \"model_version\": _model_version(spec[\"recipe\"][\"name\"], spec[\"model_sha256\"]),\n        \"feature_contract_version\": _feature_contract_version(spec[\"panel\"][\"schema_version\"], spec[\"panel\"][\"taxonomy_sha256\"],\n                                                              spec[\"feature_allowlist_sha256\"]),\n        \"repack\": {\"at_utc\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\"),\n                   \"reason\": \"release gates GPT 29/30 без переобучения: predictor, typed-поля, SHA всех файлов\",\n                   \"source_bundle_sha256sums_sha256\": _sha256_file(old_dir / \"SHA256SUMS.json\"),\n                   \"model_retrained\": False},\n    })\n    (bdir / \"bundle.json\").write_text(_dump(spec), encoding=\"utf-8\")\n    shutil.copy(code_dir / \"gas_predictor_v3.py\", bdir / \"predictor.py\")\n    shutil.copy(code_dir / \"event_panel_v3.py\", bdir / \"features_v3\" / \"event_panel_v3.py\")\n    (bdir / \"feature_contract.json\").write_text(_dump(_contract(spec, spec[\"features\"])), encoding=\"utf-8\")\n    audit = json.loads((bdir / \"audit.json\").read_text(encoding=\"utf-8\"))\n    (bdir / \"README_RU.md\").write_text(bundle_readme(spec, audit), encoding=\"utf-8\")\n    (bdir / \"SHA256SUMS.json\").unlink()\n    zpath = _pack(bdir, out_dir)\n    (out_dir / f\"{BUNDLE_DIR_NAME}.zip.sha256\").write_text(f\"{_sha256_file(zpath)}  {zpath.name}\\n\", encoding=\"utf-8\")\n    return bdir\n\n\ndef verify_repack(new_dir: Path, old_dir: Path, n: int = 5000, seed: int = 0) -> dict[str, Any]:\n    \"\"\"Новый predictor на переупакованном bundle против старого predictor на исходном: те же скоры на синтетических\n    записях (реальных строк не нужно), плюс проверки контракта. Только агрегаты.\"\"\"\n    import pandas as pd\n\n    old = _import_predictor(old_dir)\n    new = _import_predictor(new_dir)\n    b_old = old.load_bundle(old_dir)\n    b_new = new.load_bundle(new_dir)\n    rng = np.random.default_rng(seed)\n    rows = {}\n    for f in b_new[\"spec\"][\"features\"]:\n        if f[\"kind\"] == \"categorical\":\n            lv = list(f[\"levels\"]) + [None]\n            rows[f[\"name\"]] = [lv[i] for i in rng.integers(0, len(lv), n)]\n        else:\n            v = np.abs(rng.normal(0, 3, n)) * rng.choice([0.0, 1.0, 10.0], n)\n            v[rng.random(n) < 0.05] = np.nan\n            rows[f[\"name\"]] = v\n    rows[\"f_gas_n_7d\"] = rng.integers(0, 50, n).astype(float)\n    rows[\"f_gas_last\"] = rng.uniform(0, 1.3, n)\n    frame = pd.DataFrame(rows)\n    frame.insert(0, \"as_of_date\", \"2026-01-15\")\n    a = old.predict(frame, b_old)\n    b = new.predict(frame, b_new)\n    sa = np.array([np.nan if r[\"score\"] is None else r[\"score\"] for r in a])\n    sb = np.array([np.nan if r[\"score\"] is None else r[\"score\"] for r in b])\n    rep: dict[str, Any] = {\"records\": n, \"scored\": int((~np.isnan(sb)).sum()),\n                           \"same_abstain\": bool(np.array_equal(np.isnan(sa), np.isnan(sb))),\n                           \"max_abs_diff_score\": float(np.nanmax(np.abs(sa - sb))) if (~np.isnan(sb)).any() else 0.0,\n                           \"same_model_sha\": b_old[\"spec\"][\"model_sha256\"] == b_new[\"spec\"][\"model_sha256\"]}\n    rep[\"scores_identical\"] = rep[\"same_abstain\"] and rep[\"max_abs_diff_score\"] == 0.0\n    ex = json.loads((Path(new_dir) / \"example_request.json\").read_text(encoding=\"utf-8\"))\n    o = new.predict(ex, b_new)\n    rep[\"example_ok\"] = (o[0][\"decision_status\"] == \"experimental_shadow\" and o[1][\"reason_codes\"] == [\"ABOVE_THRESHOLD_AT_D\"]\n                         and o[2][\"reason_codes\"] == [\"NOT_GAS_STREAM\"])\n    rep[\"typed_fields_present\"] = all(o[0].get(k) for k in (\"model_version\", \"feature_contract_version\", \"evidence_level\",\n                                                              \"observation_cutoff_date\"))\n    for key, rec in ((\"missing_as_of_rejected\", {k: v for k, v in ex[0].items() if k != \"as_of_date\"}),\n                     (\"bad_as_of_rejected\", {**ex[0], \"as_of_date\": \"15.01.2026\"})):\n        try:\n            new.predict([rec], b_new)\n            rep[key] = False\n        except new.ContractError:\n            rep[key] = True\n    import tempfile\n\n    with tempfile.TemporaryDirectory() as tmp:\n        copy = Path(tmp) / \"b\"\n        shutil.copytree(new_dir, copy)\n        with open(copy / \"features_v3\" / \"state_taxonomy_v3.json\", \"a\", encoding=\"utf-8\") as f:\n            f.write(\" \")\n        try:\n            new.load_bundle(copy)\n            rep[\"tampered_file_rejected\"] = False\n        except new.ContractError:\n            rep[\"tampered_file_rejected\"] = True\n    rep[\"passed\"] = all(v for k, v in rep.items() if isinstance(v, bool))\n    return rep\n",
"gas_predictor_v3.py": "\"\"\"predictor.py — исполняемый скоринг газового proxy v3 для backend (shadow-режим).\n\nЧто считает: score будущего наблюдаемого строгого пересечения газового порога (1 %) снизу вверх\nв окне [D+2; D+3) по признакам, доступным к концу суток D. Это НЕ вероятность пожара и НЕ подтверждённый\nинцидент: метка — наблюдаемое событие журнала; Platt-калибровка сделана на строках с известным исходом.\n\nЗависимости: numpy, pandas, lightgbm (скоринг); polars — только для build_features.\nМодель хранится текстом LightGBM (model.txt), калибровка и контракт — в bundle.json. Pickle не используется.\n\n    from predictor import load_bundle, build_features, predict\n    bundle = load_bundle(\"gas_cross_v3_bundle\")\n    records = build_features([\"journal_extract.csv\"], \"catalogue.csv\", \"2026-09-24\", bundle)\n    results = predict(records, bundle)\n\nBackend применяет суточный бюджет (top-k) и cooldown поверх score; автоматические заявки по score выключены.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport re\nfrom datetime import date, datetime, time, timedelta\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nPASSTHROUGH = (\"as_of_date\", \"ид_канала_данных\", \"request_id\")\nNA_LEVEL = \"__NA__\"\n\n\nclass ContractError(ValueError):\n    \"\"\"Вход не соответствует feature_contract.json.\"\"\"\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef load_bundle(bundle_dir: str | Path) -> dict[str, Any]:\n    \"\"\"Загружает bundle.json и model.txt; сверяет SHA-256 всех файлов из SHA256SUMS.json и признаки модели с контрактом.\n    SHA-256 самого ZIP лежит отдельно (gas_cross_v3_bundle.zip.sha256) и проверяется при получении архива.\"\"\"\n    import lightgbm as lgb\n\n    d = Path(bundle_dir)\n    sums_path = d / \"SHA256SUMS.json\"\n    if not sums_path.exists():\n        raise ContractError(\"нет SHA256SUMS.json — bundle неполный\")\n    sums = json.loads(sums_path.read_text(encoding=\"utf-8\"))\n    bad = [name for name, sha in sums.items()\n           if not (d / name).is_file() or _sha256_bytes((d / name).read_bytes()) != sha]\n    if bad:\n        raise ContractError(f\"SHA-256 не совпадает или файла нет: {bad}\")\n    spec = json.loads((d / \"bundle.json\").read_text(encoding=\"utf-8\"))\n    model_bytes = (d / \"model.txt\").read_bytes()\n    if _sha256_bytes(model_bytes) != spec[\"model_sha256\"]:\n        raise ContractError(\"SHA-256 model.txt не совпадает с bundle.json\")\n    booster = lgb.Booster(model_str=model_bytes.decode(\"utf-8\"))\n    names = [f[\"name\"] for f in spec[\"features\"]]\n    if booster.num_feature() != len(names) or booster.feature_name() != names:\n        raise ContractError(\"признаки модели не совпадают с контрактом bundle.json\")\n    return {\"spec\": spec, \"booster\": booster, \"dir\": d}\n\n\n# ---------------------------------------------------------------------------\n# Признаки: тот же причинный код, что строил обучающую панель (ноутбук 14)\n# ---------------------------------------------------------------------------\ndef _panel_module(bundle: dict[str, Any]) -> Any:\n    path = bundle[\"dir\"] / \"features_v3\" / \"event_panel_v3.py\"\n    spec = importlib.util.spec_from_file_location(\"gas_bundle_event_panel_v3\", path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef build_features(journal_files: list[str | Path], catalogue_file: str | Path, as_of_date: str | date,\n                   bundle: dict[str, Any], channels: list[str] | None = None,\n                   min_history_days: int | None = None) -> Any:\n    \"\"\"Причинные v3-признаки на конец суток as_of_date (D) для каналов из выгрузки журнала.\n\n    journal_files — CSV/parquet с колонками журнала (ид_события, ид_канала_данных, дата, время, тревожное,\n    значение_датчика); catalogue_file — справочник каналов. События позже D отбрасываются до сборки.\n    История: ≥ bundle.json → history_days_required суток до D (иначе «неограниченные» признаки — давность\n    последней смены состояния/последнего onset, последнее значение — могут стать null и разойтись с train).\n    Если самая ранняя дата в переданной выгрузке позже D − history_days_required — ContractError\n    (INSUFFICIENT_HISTORY): score не выдаётся. Рекомендуется передавать всю доступную историю газовых каналов:\n    только тогда давность последней смены состояния/onset и последнее значение совпадают с обучающей панелью.\n    Возвращает pandas.DataFrame: as_of_date, ид_канала_данных и признаки контракта (по одной строке на канал,\n    у которого были события в [D−6; D]).\n    \"\"\"\n    import polars as pl\n\n    spec = bundle[\"spec\"]\n    ep = _panel_module(bundle)\n    d = as_of_date if isinstance(as_of_date, date) else date.fromisoformat(str(as_of_date))\n    flt = pl.col(\"дата\") <= d.isoformat()\n    if channels is not None:\n        flt = flt & pl.col(\"ид_канала_данных\").is_in([str(c) for c in channels])\n    files = [Path(p) for p in journal_files]\n    need = int(spec.get(\"history_days_required\", 0) if min_history_days is None else min_history_days)\n    first = min((ep._scan_source(pl, f).filter(flt).select(pl.col(\"дата\").min()).collect().item() for f in files),\n                key=lambda x: x or \"9999-12-31\")\n    if first is None:\n        raise ContractError(\"INSUFFICIENT_HISTORY: в выгрузке нет событий до D\")\n    if date.fromisoformat(str(first)[:10]) > d - timedelta(days=need):\n        raise ContractError(f\"INSUFFICIENT_HISTORY: история с {str(first)[:10]}, нужно не позже \"\n                            f\"{(d - timedelta(days=need)).isoformat()} ({need} суток до D)\")\n    panel, _ = ep.build_event_panel(pl, files, Path(catalogue_file), bundle[\"dir\"] / \"features_v3\" / \"state_taxonomy_v3.json\",\n                                    config=spec[\"panel_config\"], log=lambda *a: None, channel_filter=flt,\n                                    data_end=datetime.combine(d, time(23, 59, 59)))\n    rows = panel.filter(pl.col(\"d_cutoff_date\") == d)\n    raw_ids = set()\n    for f in files:\n        raw_ids.update(ep._scan_source(pl, f).filter(flt).select(\"ид_канала_данных\").unique().collect()\n                       [\"ид_канала_данных\"].to_list())\n    keys = ep.key_map(pl, raw_ids, \"d_channel_key\")\n    rows = rows.join(keys, on=\"d_channel_key\", how=\"left\").rename({\"raw\": \"ид_канала_данных\"})\n    names = [f[\"name\"] for f in spec[\"features\"]]\n    out = rows.select([pl.lit(d.isoformat()).alias(\"as_of_date\"), \"ид_канала_данных\"] + names)\n    return out.to_pandas()\n\n\n# ---------------------------------------------------------------------------\n# Скоринг\n# ---------------------------------------------------------------------------\ndef _frame(records: Any, spec: dict[str, Any]) -> tuple[Any, Any]:\n    import pandas as pd\n\n    df = records if isinstance(records, pd.DataFrame) else pd.DataFrame(list(records))\n    df = df.reset_index(drop=True)\n    if \"as_of_date\" not in df.columns:\n        raise ContractError(\"нет обязательного поля as_of_date (YYYY-MM-DD, конец суток D)\")\n    for i, v in enumerate(df[\"as_of_date\"].tolist()):\n        text = v.isoformat()[:10] if isinstance(v, (date, datetime)) else str(v)\n        try:\n            if not re.fullmatch(r\"\\d{4}-\\d{2}-\\d{2}\", text):\n                raise ValueError\n            date.fromisoformat(text)\n        except ValueError as exc:\n            raise ContractError(f\"as_of_date в строке {i}: ожидается дата YYYY-MM-DD, получено {v!r}\") from exc\n    df[\"as_of_date\"] = [v.isoformat()[:10] if isinstance(v, (date, datetime)) else str(v) for v in df[\"as_of_date\"]]\n    names = [f[\"name\"] for f in spec[\"features\"]]\n    missing = [n for n in names if n not in df.columns]\n    if missing:\n        raise ContractError(f\"нет обязательных признаков: {missing[:10]}{' …' if len(missing) > 10 else ''}\")\n    extra = [c for c in df.columns if c not in names and c not in PASSTHROUGH]\n    if extra:\n        raise ContractError(f\"лишние поля (не из контракта): {extra[:10]}\")\n    x = pd.DataFrame(index=df.index)\n    for f in spec[\"features\"]:\n        col = df[f[\"name\"]]\n        if f[\"kind\"] == \"categorical\":\n            v = col.astype(object).where(col.notna(), NA_LEVEL).astype(str)\n            v = v.where(v.isin(f[\"levels\"]), None)                  # неизвестный уровень → пропуск (как в train)\n            x[f[\"name\"]] = pd.Categorical(v, categories=f[\"levels\"])\n        else:\n            try:\n                x[f[\"name\"]] = pd.to_numeric(col, errors=\"raise\").astype(\"float32\")\n            except (TypeError, ValueError) as exc:\n                raise ContractError(f\"признак {f['name']}: ожидается число или null\") from exc\n    return df, x\n\n\ndef _calibrate(raw: np.ndarray, platt: dict[str, float]) -> np.ndarray:\n    p = np.clip(raw, 1e-6, 1 - 1e-6)\n    z = np.log(p / (1 - p))\n    return 1.0 / (1.0 + np.exp(-(platt[\"coef\"] * z + platt[\"intercept\"])))\n\n\ndef eligibility(df: Any, spec: dict[str, Any]) -> tuple[np.ndarray, list[list[str]]]:\n    \"\"\"Runtime eligibility на конец D (как gas_elig в панели 14): газовый поток в [D−6; D] и последнее показание < порога.\"\"\"\n    import pandas as pd\n\n    thr = float(spec[\"eligibility\"][\"gas_threshold_percent\"])\n    n7 = pd.to_numeric(df[\"f_gas_n_7d\"], errors=\"coerce\").to_numpy(dtype=float)\n    last = pd.to_numeric(df[\"f_gas_last\"], errors=\"coerce\").to_numpy(dtype=float)\n    ok = np.zeros(len(df), dtype=bool)\n    reasons: list[list[str]] = []\n    for i in range(len(df)):\n        if not (n7[i] > 0):\n            reasons.append([\"NOT_GAS_STREAM\"])\n        elif math.isnan(last[i]):\n            reasons.append([\"NO_GAS_READING_AT_D\"])\n        elif last[i] >= thr:\n            reasons.append([\"ABOVE_THRESHOLD_AT_D\"])\n        else:\n            ok[i] = True\n            reasons.append([])\n    return ok, reasons\n\n\ndef predict(records: Any, bundle: dict[str, Any]) -> list[dict[str, Any]]:\n    \"\"\"records: список dict или pandas.DataFrame с признаками контракта (+ as_of_date, ид_канала_данных, request_id).\n    Для неподходящих строк score = null и decision_status = abstain с причиной.\"\"\"\n    spec = bundle[\"spec\"]\n    df, x = _frame(records, spec)\n    ok, reasons = eligibility(df, spec)\n    raw = np.full(len(df), np.nan)\n    if ok.any():\n        raw[ok] = bundle[\"booster\"].predict(x[ok])\n    cal = np.where(ok, _calibrate(np.nan_to_num(raw, nan=0.5), spec[\"platt\"]), np.nan)\n    w = spec[\"window\"]\n    out = []\n    for i in range(len(df)):\n        d = date.fromisoformat(str(df[\"as_of_date\"].iloc[i])[:10])\n        item = {\n            \"request_index\": i,\n            \"as_of_date\": d.isoformat(),\n            \"observation_cutoff_date\": d.isoformat(),      # признаки — по событиям до конца этих суток\n            \"target_code\": spec[\"target_code\"],\n            \"model_version\": spec[\"model_version\"],\n            \"feature_contract_version\": spec[\"feature_contract_version\"],\n            \"evidence_level\": spec[\"evidence_level\"],\n            \"incident_type\": spec[\"incident_type\"],\n            \"score\": float(cal[i]) if ok[i] else None,\n            \"raw_score\": float(raw[i]) if ok[i] else None,\n            \"score_kind\": spec[\"score_kind\"],\n            \"decision_status\": spec[\"decision_status\"] if ok[i] else \"abstain\",\n            \"reason_codes\": reasons[i],\n            \"window_start\": (d + timedelta(days=w[\"start_offset_days\"])).isoformat(),\n            \"window_end_exclusive\": (d + timedelta(days=w[\"end_offset_days_exclusive\"])).isoformat(),\n            \"evidence_scope\": spec[\"evidence_scope\"],\n            \"operational_ready\": spec[\"operational_ready\"],\n            \"not_a_confirmed_incident\": True,\n            \"not_fire_probability\": True,\n        }\n        for k in (\"ид_канала_данных\", \"request_id\"):\n            if k in df.columns:\n                item[k] = None if df[k].isna().iloc[i] else str(df[k].iloc[i])\n        out.append(item)\n    return out\n",
"event_panel_v3.py": "\"\"\"Событийная панель v3: state machine по сырому журналу → панель (канал × сутки D) с раздельными целями.\n\nСпецификация согласована в переписке Claude ↔ GPT (Drive, документы 05 → 12) и docs/EVENT_PANEL_V3_RU.md.\nГлавные правила:\n- одна секунда канала = неупорядоченное множество значений; порядок строк CSV и ид_события порядком не считаются;\n- признаки на конец D — только прошлое; метка окна [D+2; D+3) может смотреть вперёд не дальше D+4 (L = 24 ч);\n- eligibility на конец D задаётся только состоянием на конец D; старт в D+1 — competing outcome, а не фильтр;\n- неоднозначное → null с причиной, а не 0;\n- цели разные и не смешиваются: T2a (onset «Неисправен» = потеря связи), T2b (потеря связи ≥ τ),\n  T2c (onset «Обесточен»), T2c_sustained, T1a (технические/диапазонные значения), T4 (газ ≥ 1 %).\nНичто здесь не является подтверждённым физическим отказом.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom datetime import date, datetime, timedelta\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nSCHEMA_VERSION = \"3.0\"\nEVENT_COLUMNS = [\"ид_события\", \"ид_канала_данных\", \"дата\", \"время\", \"тревожное\", \"значение_датчика\"]\nCATALOGUE_COLUMNS = [\"ид_канала_данных\", \"тип_инж_системы\", \"тип_датчика\", \"ид_объект\"]\nKEY_SALT = \"ldt-misis-panel-v3:\"\n\nDEFAULT_CONFIG: dict[str, Any] = {\n    \"tau_minutes\": 5,                 # порог «устойчивого» эпизода; чувствительность 1/5/30/60 — отдельными прогонами\n    \"confirm_hours\": 24,              # L: самое позднее подтверждение метки, ч\n    \"active_lookback_days\": 7,        # строка (канал, D) есть, если канал слал события в [D-6; D]\n    \"tech_clean_days\": 7,             # T1a: onset только после ≥ 7 суток без технических значений\n    \"numeric_history_days\": 30,       # T1a: канал должен слать числа в последние 30 суток\n    \"window_offsets_days\": [7, 30],   # окна скользящих признаков\n    \"service_as_unknown\": True,       # «Неопределен» = U для связи (основная версия); False — sensitivity (GPT 16/Q1)\n    \"power_ok_from_function\": False,  # sensitivity: сообщения функции канала (норма/событие/число) считаются O питания\n}\n\n# заранее объявленные sensitivity-варианты разметки (выбор между ними не делается по validation-метрикам)\nSENSITIVITY_VARIANTS = {\"service_neutral\": {\"service_as_unknown\": False},\n                        \"power_ok_from_function\": {\"power_ok_from_function\": True}}\n\nLINK_STATES = (\"O\", \"F\", \"A\")  # ok / fault / ambiguous\nTARGETS = [\"target_t2a_link_onset\", \"target_t2b_link_sustained\", \"target_t2c_power_onset\",\n           \"target_t2c_power_sustained\", \"target_t1a_tech_value\", \"target_t4_gas_cross\"]\nBASE_FEATURES = [\"n_events\", \"n_alarm\", \"n_link_fault\", \"n_power_off\", \"n_service\", \"n_tech_like\", \"n_numeric\",\n                 \"n_multi_value_seconds\", \"num_mean\", \"num_min\", \"num_max\", \"num_std\", \"num_last\", \"gas_max\", \"gas_last\",\n                 \"link_state\", \"power_state\", \"d_weekday\", \"d_month\"]\n\n\n# ---------------------------------------------------------------------------\n# 0. Таксономия и ключи\n# ---------------------------------------------------------------------------\ndef load_taxonomy(path: Path) -> dict[str, Any]:\n    tax = json.loads(Path(path).read_text(encoding=\"utf-8\"))\n    seen: dict[str, str] = {}\n    for cls, values in tax[\"text_classes\"].items():\n        for v in values:\n            if v in seen and seen[v] != cls:\n                raise ValueError(f\"значение {v!r} в двух классах: {seen[v]} и {cls}\")\n            seen[v] = cls\n    tax[\"_value_to_class\"] = seen\n    tax[\"_sha256\"] = hashlib.sha256(Path(path).read_bytes()).hexdigest()\n    return tax\n\n\ndef pseudo_key(raw: str | None) -> str | None:\n    if raw is None:\n        return None\n    return hashlib.sha256((KEY_SALT + str(raw)).encode(\"utf-8\")).hexdigest()[:16]\n\n\ndef key_map(pl: Any, ids: Iterable[str], name: str) -> Any:\n    ids = sorted({str(i) for i in ids if i is not None})\n    return pl.DataFrame({\"raw\": ids, name: [pseudo_key(i) for i in ids]})\n\n\n# ---------------------------------------------------------------------------\n# 1. Классификация событий\n# ---------------------------------------------------------------------------\ndef classify_events(pl: Any, frame: Any, tax: dict[str, Any], catalogue: Any) -> Any:\n    \"\"\"frame: сырые колонки журнала (строки). Возвращает ch, t, v, alarm, num, cls, is_gas, тип_*.\"\"\"\n    num_cfg, date_cfg = tax[\"numeric\"], tax[\"date_like\"]\n    value = pl.col(\"значение_датчика\").fill_null(\"\")\n    num = value.str.replace(\",\", \".\", literal=True).cast(pl.Float64, strict=False)\n    t = pl.concat_str([\"дата\", \"время\"], separator=\" \").str.strptime(pl.Datetime(\"us\"), \"%Y-%m-%d %H:%M:%S\", strict=False)\n    out = (\n        frame.join(catalogue, on=\"ид_канала_данных\", how=\"left\")\n        .with_columns(\n            pl.col(\"ид_канала_данных\").alias(\"ch\"), t.alias(\"t\"), value.alias(\"v\"), num.alias(\"num\"),\n            pl.col(\"тревожное\").fill_null(\"\").str.to_lowercase().is_in([\"t\", \"true\"]).alias(\"alarm\"),\n            (pl.col(\"тип_датчика\") == num_cfg[\"gas_sensor_type\"]).fill_null(False).alias(\"is_gas\"),\n        )\n        .filter(pl.col(\"t\").is_not_null())\n    )\n    is_date = pl.col(\"v\").str.contains(date_cfg[\"regex\"])\n    parsed_date = pl.col(\"v\").str.slice(0, 19).str.strptime(pl.Datetime(\"us\"), \"%d.%m.%Y %H:%M:%S\", strict=False)\n    tol = int(date_cfg[\"duplicate_timestamp_tolerance_seconds\"])\n    text_cls = pl.col(\"v\").replace_strict(tax[\"_value_to_class\"], default=\"unknown_value\", return_dtype=pl.String)\n    cls = (\n        pl.when(pl.col(\"num\").is_not_null() & pl.col(\"num\").is_in(num_cfg[\"technical_value_candidates\"])).then(pl.lit(\"tech_value\"))\n        .when(pl.col(\"num\").is_not_null() & pl.col(\"is_gas\")\n              & ((pl.col(\"num\") < num_cfg[\"gas_out_of_range\"][\"below\"]) | (pl.col(\"num\") > num_cfg[\"gas_out_of_range\"][\"above\"])))\n        .then(pl.lit(\"gas_out_of_range\"))\n        .when(pl.col(\"num\").is_not_null()).then(pl.lit(\"numeric\"))\n        .when(is_date & pl.col(\"v\").str.starts_with(date_cfg[\"clock_1970_prefix\"])).then(pl.lit(\"clock_1970\"))\n        .when(is_date & ((parsed_date - pl.col(\"t\")).dt.total_seconds().abs() <= tol)).then(pl.lit(\"dup_timestamp\"))\n        .when(is_date).then(pl.lit(\"date_other\"))\n        .when(pl.col(\"v\") == \"\").then(pl.lit(\"empty_value\"))\n        .otherwise(text_cls)\n    )\n    return out.with_columns(cls.alias(\"cls\")).select(\n        \"ch\", \"t\", \"v\", \"alarm\", \"num\", \"cls\", \"is_gas\", \"тип_датчика\", \"тип_инж_системы\", \"ид_объект\")\n\n\n# ---------------------------------------------------------------------------\n# 2. Секундный уровень и точки смены состояния (RLE)\n# ---------------------------------------------------------------------------\ndef second_table(pl: Any, ev: Any, tax: dict[str, Any]) -> Any:\n    \"\"\"Одна строка = (канал, секунда). Порядок внутри секунды не используется.\n    Для каждого потока (связь, питание) секунда: F — только отказ, O — только рабочее состояние того же потока,\n    A — отказ и O вместе или любое неизвестное (U) этого потока; null — секунда не несёт сведений о потоке.\"\"\"\n    streams = tax[\"stream_mapping\"][\"streams\"]\n    aggs = [pl.col(\"v\").n_unique().alias(\"n_values\"), (pl.col(\"cls\") == \"link_fault\").any().alias(\"has_fault\")]\n    for name, st in streams.items():\n        aggs += [pl.col(\"cls\").is_in(st[\"fault\"]).any().alias(f\"_{name}_f\"),\n                 pl.col(\"cls\").is_in(st[\"ok\"]).any().alias(f\"_{name}_o\"),\n                 pl.col(\"cls\").is_in(st[\"unknown\"]).any().alias(f\"_{name}_u\")]\n    sec = ev.group_by(\"ch\", \"t\").agg(*aggs)\n    cols = []\n    for name in streams:\n        f, o, u = pl.col(f\"_{name}_f\"), pl.col(f\"_{name}_o\"), pl.col(f\"_{name}_u\")\n        override = bool(streams[name].get(\"unknown_overrides\", True))\n        u_wins = u if override else (u & ~f & ~o)\n        cols.append(pl.when(f & o).then(pl.lit(\"A\")).when(u_wins).then(pl.lit(\"A\"))\n                    .when(f).then(pl.lit(\"F\")).when(o).then(pl.lit(\"O\")).otherwise(None).alias(f\"{name}_cls\"))\n        cols.append(pl.when(f & o).then(pl.lit(\"conflict\")).when(u_wins).then(pl.lit(\"unknown\")).otherwise(None)\n                    .alias(f\"{name}_a_cause\"))\n    return sec.with_columns(cols).drop([c for c in sec.columns if c.startswith(\"_\")])\n\n\ndef change_points(pl: Any, sec: Any, cls_col: str) -> Any:\n    \"\"\"RLE решающих секунд: строки, где класс сменился (по каналу, по времени).\"\"\"\n    d = sec.filter(pl.col(cls_col).is_not_null()).select(\"ch\", \"t\", pl.col(cls_col).alias(\"cls\")).sort(\"ch\", \"t\")\n    return rle(pl, d)\n\n\ndef rle(pl: Any, d: Any) -> Any:\n    d = d.sort(\"ch\", \"t\")\n    prev = pl.col(\"cls\").shift(1).over(\"ch\")\n    return d.filter(prev.is_null() | (pl.col(\"cls\") != prev)).select(\"ch\", \"t\", \"cls\")\n\n\n# ---------------------------------------------------------------------------\n# 3. Эпизоды\n# ---------------------------------------------------------------------------\ndef episodes(pl: Any, cp: Any, data_end: datetime, tau_minutes: float) -> Any:\n    \"\"\"Эпизод = RLE-блок класса F. onset_type: strict (перед ним однозначное O) или ambiguous.\n    Длительность: до следующей точки смены. Следующая O → exact; A или конец данных → цензура (нижняя граница).\n    sustained = 1 только при однозначном закрытии (O того же потока) после t0 + τ: повторы «Неисправен» и тишина\n    непрерывность не доказывают (GPT 14, C7); незакрытый эпизод → null.\"\"\"\n    cp = cp.sort(\"ch\", \"t\").with_columns(\n        pl.col(\"cls\").shift(1).over(\"ch\").alias(\"prev_cls\"),\n        pl.col(\"cls\").shift(-1).over(\"ch\").alias(\"next_cls\"),\n        pl.col(\"t\").shift(-1).over(\"ch\").alias(\"next_t\"),\n    )\n    tau_s = tau_minutes * 60.0\n    ep = cp.filter(pl.col(\"cls\") == \"F\").with_columns(\n        pl.when(pl.col(\"prev_cls\") == \"O\").then(pl.lit(\"strict\")).otherwise(pl.lit(\"ambiguous\")).alias(\"onset_type\"),\n        (pl.col(\"next_cls\") == \"O\").fill_null(False).alias(\"exact\"),\n        pl.col(\"next_t\").fill_null(pl.lit(data_end)).alias(\"end_t\"),\n    ).with_columns(\n        (pl.col(\"end_t\") - pl.col(\"t\")).dt.total_seconds().cast(pl.Float64).alias(\"duration_s\"),\n    ).with_columns(\n        pl.when(pl.col(\"exact\") & (pl.col(\"duration_s\") >= tau_s)).then(pl.lit(1, pl.Int8))\n        .when(pl.col(\"exact\")).then(pl.lit(0, pl.Int8))\n        .otherwise(None).alias(\"sustained\"),\n    )\n    return ep.select(\"ch\", pl.col(\"t\").alias(\"onset_t\"), \"onset_type\", \"exact\", \"end_t\", \"duration_s\", \"sustained\")\n\n\n# ---------------------------------------------------------------------------\n# 4. Дневные агрегаты\n# ---------------------------------------------------------------------------\nCOUNT_CLASSES = [\"link_fault\", \"power_off\", \"power_on\", \"power_degraded\", \"device_off\", \"service\",\n                 \"undefined_measurement\", \"object_summary\", \"security_mode\", \"flood\", \"event_alarm\", \"normal\",\n                 \"numeric\", \"tech_value\", \"gas_out_of_range\", \"clock_1970\", \"dup_timestamp\", \"date_other\",\n                 \"unknown_value\", \"empty_value\"]\n\n\ndef daily_aggregates(pl: Any, ev: Any, sec: Any, tax: dict[str, Any]) -> Any:\n    thr = tax[\"numeric\"][\"gas_threshold_percent\"]\n    low = tax[\"numeric\"][\"gas_band_low\"]\n    warn = tax[\"numeric\"][\"gas_warning_percent\"]\n    # порядок внутри группы = время; в одной секунде значения не упорядочены — берём детерминированно (по значению)\n    ev = ev.sort(\"ch\", \"t\", \"num\", nulls_last=False, maintain_order=True).with_columns(pl.col(\"t\").dt.date().alias(\"day\"))\n    numeric = pl.col(\"cls\") == \"numeric\"\n    gas = numeric & pl.col(\"is_gas\")\n    agg = ev.group_by(\"ch\", \"day\").agg(\n        pl.len().alias(\"n_events\"),\n        pl.col(\"alarm\").sum().alias(\"n_alarm\"),\n        *[(pl.col(\"cls\") == c).sum().alias(f\"n_{c}\") for c in COUNT_CLASSES],\n        pl.col(\"num\").filter(numeric).mean().alias(\"num_mean\"),\n        pl.col(\"num\").filter(numeric).min().alias(\"num_min\"),\n        pl.col(\"num\").filter(numeric).max().alias(\"num_max\"),\n        pl.col(\"num\").filter(numeric).std().alias(\"num_std\"),\n        pl.col(\"num\").filter(numeric).last().alias(\"num_last\"),\n        pl.col(\"num\").filter(gas).max().alias(\"gas_max\"),\n        pl.col(\"num\").filter(gas).count().alias(\"gas_n\"),\n        (gas & (pl.col(\"num\") >= low) & (pl.col(\"num\") < thr)).sum().alias(\"gas_band_n\"),\n        (gas & (pl.col(\"num\") >= thr)).sum().alias(\"gas_ge_thr_n\"),\n        (gas & (pl.col(\"num\") >= warn)).sum().alias(\"gas_ge_warn_n\"),\n        pl.col(\"num\").filter(gas).last().alias(\"gas_last\"),\n        # последняя газовая секунда суток содержит значения по обе стороны порога → сторона на конец D неизвестна (GPT 42)\n        ((pl.col(\"num\").filter(gas & (pl.col(\"t\") == pl.col(\"t\").filter(gas).max())).min() < thr)\n         & (pl.col(\"num\").filter(gas & (pl.col(\"t\") == pl.col(\"t\").filter(gas).max())).max() >= thr)).alias(\"gas_last_mixed\"),\n    )\n    s = sec.with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n        (pl.col(\"n_values\") > 1).sum().alias(\"n_multi_value_seconds\"),\n        (pl.col(\"link_cls\") == \"F\").sum().alias(\"n_link_fault_seconds\"),\n        (pl.col(\"link_cls\") == \"A\").sum().alias(\"n_link_ambiguous_seconds\"),\n        (pl.col(\"power_cls\") == \"F\").sum().alias(\"n_power_off_seconds\"),\n    )\n    return agg.join(s, on=[\"ch\", \"day\"], how=\"left\")\n\n\ndef gas_crossings(pl: Any, ev: Any, carry: Any, thr: float) -> tuple[Any, Any]:\n    \"\"\"Строгое пересечение порога на секундном уровне (секунда = множество значений без порядка, как для состояний):\n    strict — все показания секунды ≥ thr, а все показания предыдущей секунды того же канала < thr;\n    неоднозначно (cross_unknown) — в секунде есть значения по обе стороны порога, либо предыдущая секунда такая,\n    либо предыдущего показания нет. Раньше показания одной секунды сортировались по значению, и пара 0 / 2,55\n    в одну секунду давала ложный strict crossing (GPT 37).\n    carry: последняя газовая секунда каждого канала из предыдущего года (ch, t, mn, mx).\"\"\"\n    g = (ev.filter((pl.col(\"cls\") == \"numeric\") & pl.col(\"is_gas\"))\n         .group_by(\"ch\", \"t\").agg(pl.col(\"num\").min().alias(\"mn\"), pl.col(\"num\").max().alias(\"mx\")))\n    both = pl.concat([carry.select(\"ch\", \"t\", \"mn\", \"mx\").with_columns(pl.lit(True).alias(\"_carry\")),\n                      g.with_columns(pl.lit(False).alias(\"_carry\"))], how=\"vertical\").sort(\"ch\", \"t\", \"_carry\",\n                                                                                          descending=[False, False, True])\n    pmn, pmx = pl.col(\"mn\").shift(1).over(\"ch\"), pl.col(\"mx\").shift(1).over(\"ch\")\n    straddle = (pl.col(\"mn\") < thr) & (pl.col(\"mx\") >= thr)\n    prev_straddle = (pmn < thr) & (pmx >= thr)\n    x = both.with_columns(\n        ((pl.col(\"mn\") >= thr) & (pmx < thr)).fill_null(False).alias(\"cross\"),\n        ((pl.col(\"mx\") >= thr) & (pmx.is_null() | straddle | ((pl.col(\"mn\") >= thr) & prev_straddle.fill_null(False))))\n        .fill_null(False).alias(\"cross_unknown_prev\"))\n    x = x.filter(~pl.col(\"_carry\"))\n    daily = x.with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n        pl.col(\"cross\").sum().alias(\"gas_cross_n\"), pl.col(\"cross_unknown_prev\").sum().alias(\"gas_cross_unknown_n\"))\n    new_carry = (pl.concat([carry.select(\"ch\", \"t\", \"mn\", \"mx\"), g], how=\"vertical\").sort(\"ch\", \"t\")\n                 .group_by(\"ch\", maintain_order=True).agg(pl.col(\"t\").last(), pl.col(\"mn\").last(), pl.col(\"mx\").last()))\n    return daily, new_carry\n\n\n# ---------------------------------------------------------------------------\n# 5. Сборка панели\n# ---------------------------------------------------------------------------\ndef _shift_days(pl: Any, frame: Any, cols: list[str], offset: int, suffix: str) -> Any:\n    \"\"\"Факты дня X → строки D = X − offset.\"\"\"\n    return frame.select(\"ch\", (pl.col(\"day\") - pl.duration(days=offset)).alias(\"day\"),\n                        *[pl.col(c).alias(f\"{c}{suffix}\") for c in cols])\n\n\ndef build_panel_from_parts(pl: Any, daily: Any, link_cp: Any, power_cp: Any, gas_daily: Any,\n                           meta: Any, data_end: datetime, config: dict[str, Any]) -> tuple[Any, dict[str, Any]]:\n    tau = float(config[\"tau_minutes\"])\n    L = int(config[\"confirm_hours\"])\n    lookback = int(config[\"active_lookback_days\"])\n    data_end_day = data_end.date()\n\n    link_ep = episodes(pl, link_cp, data_end, tau)\n    power_ep = episodes(pl, power_cp, data_end, tau)\n\n    daily = daily.join(gas_daily, on=[\"ch\", \"day\"], how=\"left\").with_columns(\n        pl.col(\"gas_cross_n\").fill_null(0), pl.col(\"gas_cross_unknown_n\").fill_null(0))\n\n    # грид: (канал, D), если канал слал события в [D-lookback+1; D]\n    grid = (daily.select(\"ch\", \"day\")\n            .with_columns(pl.int_ranges(0, lookback).alias(\"_k\")).explode(\"_k\")\n            .with_columns((pl.col(\"day\") + pl.duration(days=pl.col(\"_k\"))).alias(\"day\"))\n            .filter(pl.col(\"day\") <= pl.lit(data_end_day))\n            .select(\"ch\", \"day\").unique().sort(\"ch\", \"day\"))\n\n    count_cols = [c for c in daily.columns if c.startswith(\"n_\") or c in (\"gas_n\", \"gas_band_n\", \"gas_ge_thr_n\",\n                                                                               \"gas_ge_warn_n\", \"gas_cross_n\")]\n    panel = grid.join(daily, on=[\"ch\", \"day\"], how=\"left\").with_columns(\n        [pl.col(c).fill_null(0) for c in count_cols]).sort(\"ch\", \"day\")\n\n    # эпизоды по дням: onset-день (для меток и признаков) и день завершения (для признаков «прошлых эпизодов»)\n    def per_day(ep: Any, prefix: str, cp: Any) -> tuple[Any, Any, Any]:\n        on = ep.with_columns(pl.col(\"onset_t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            (pl.col(\"onset_type\") == \"strict\").sum().alias(f\"{prefix}_onset_strict\"),\n            (pl.col(\"onset_type\") == \"ambiguous\").sum().alias(f\"{prefix}_onset_ambiguous\"),\n            ((pl.col(\"onset_type\") == \"strict\") & (pl.col(\"sustained\") == 1)).sum().alias(f\"{prefix}_onset_sustained\"),\n            ((pl.col(\"onset_type\") == \"strict\") & (pl.col(\"sustained\") == 0)).sum().alias(f\"{prefix}_onset_transient\"),\n            ((pl.col(\"onset_type\") == \"strict\") & pl.col(\"sustained\").is_null()).sum().alias(f\"{prefix}_onset_censored\"),\n        )\n        done = ep.filter(pl.col(\"exact\")).with_columns(pl.col(\"end_t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            (pl.col(\"sustained\") == 1).sum().alias(f\"{prefix}_completed_long\"),\n            (pl.col(\"sustained\") == 0).sum().alias(f\"{prefix}_completed_short\"),\n        )\n        # неоднозначные секунды (A-блоки): переход мог быть скрыт — окно с таким блоком не отрицательное (GPT 14, C2/C12)\n        amb = cp.filter(pl.col(\"cls\") == \"A\").with_columns(pl.col(\"t\").dt.date().alias(\"day\")).group_by(\"ch\", \"day\").agg(\n            pl.len().cast(pl.Int64).alias(f\"{prefix}_ambiguous_block\"))\n        return on, done, amb\n\n    link_on, link_done, link_amb = per_day(link_ep, \"link\", link_cp)\n    power_on, power_done, power_amb = per_day(power_ep, \"power\", power_cp)\n    for part in (link_on, link_done, power_on, power_done, link_amb, power_amb):\n        cols = [c for c in part.columns if c not in (\"ch\", \"day\")]\n        panel = panel.join(part, on=[\"ch\", \"day\"], how=\"left\").with_columns([pl.col(c).fill_null(0) for c in cols])\n\n    # скользящие признаки (только прошлое, включая D)\n    roll_src = [\"n_events\", \"n_alarm\", \"n_link_fault\", \"n_power_off\", \"n_service\", \"n_device_off\",\n                \"n_undefined_measurement\", \"n_tech_value\", \"n_gas_out_of_range\", \"n_clock_1970\", \"n_dup_timestamp\",\n                \"n_date_other\", \"n_unknown_value\", \"n_flood\", \"n_event_alarm\", \"n_numeric\", \"n_multi_value_seconds\",\n                \"n_link_fault_seconds\", \"n_link_ambiguous_seconds\", \"n_power_off_seconds\",\n                \"link_onset_strict\", \"link_onset_ambiguous\", \"power_onset_strict\",\n                \"link_completed_long\", \"link_completed_short\", \"power_completed_long\", \"power_completed_short\",\n                \"gas_n\", \"gas_band_n\", \"gas_ge_thr_n\", \"gas_ge_warn_n\", \"gas_cross_n\"]\n    panel = panel.with_columns((pl.col(\"n_events\") > 0).cast(pl.Int32).alias(\"active_day\"),\n                               (pl.col(\"n_tech_value\") + pl.col(\"n_gas_out_of_range\") + pl.col(\"n_undefined_measurement\"))\n                               .alias(\"n_tech_like\"),\n                               (pl.col(\"n_numeric\") + pl.col(\"n_undefined_measurement\") + pl.col(\"n_tech_value\")\n                                + pl.col(\"n_gas_out_of_range\")).alias(\"n_measurement\"))\n    roll_exprs = []\n    for w in config[\"window_offsets_days\"]:\n        for c in roll_src + [\"active_day\", \"n_tech_like\", \"n_measurement\"]:\n            # polars < 1.3x не принимает null во входе rolling_*_by: счётчики без событий = 0\n            roll_exprs.append(pl.col(c).fill_null(0).rolling_sum_by(\"day\", window_size=f\"{w}d\", closed=\"right\")\n                              .over(\"ch\").alias(f\"f_{c}_{w}d\"))\n        # максимум газа: дни без газовых чисел → −∞ во входе, окно без чисел → null на выходе\n        gmax = pl.col(\"gas_max\").fill_null(float(\"-inf\")).rolling_max_by(\"day\", window_size=f\"{w}d\", closed=\"right\").over(\"ch\")\n        roll_exprs.append(pl.when(gmax == float(\"-inf\")).then(None).otherwise(gmax).alias(f\"f_gas_max_{w}d\"))\n    panel = panel.sort(\"ch\", \"day\").with_columns(roll_exprs)\n    # последнее газовое/числовое значение на конец D (forward fill по грид-строкам канала)\n    panel = panel.with_columns(pl.col(\"gas_last\").forward_fill().over(\"ch\").alias(\"f_gas_last\"),\n                               pl.col(\"num_last\").forward_fill().over(\"ch\").alias(\"f_num_last\"),\n                               pl.when(pl.col(\"gas_last\").is_not_null()).then(pl.col(\"gas_last_mixed\").fill_null(False))\n                               .forward_fill().over(\"ch\").fill_null(False).alias(\"_gas_last_mixed\"))\n\n    # состояния на конец D (as-of по точкам смены)\n    cutoff = (pl.col(\"day\").cast(pl.Datetime(\"us\")) + pl.duration(days=1) - pl.duration(microseconds=1))\n    panel = panel.with_columns(cutoff.alias(\"_cutoff_t\")).sort(\"_cutoff_t\")\n    for name, cp in ((\"link\", link_cp), (\"power\", power_cp)):\n        c = cp.select(\"ch\", pl.col(\"t\").alias(f\"_{name}_t\"), pl.col(\"cls\").alias(f\"{name}_state\")).sort(f\"_{name}_t\")\n        panel = panel.join_asof(c, left_on=\"_cutoff_t\", right_on=f\"_{name}_t\", by=\"ch\", strategy=\"backward\")\n        panel = panel.with_columns(\n            ((pl.col(\"_cutoff_t\") - pl.col(f\"_{name}_t\")).dt.total_seconds() / 86400.0).alias(f\"f_{name}_state_age_days\"))\n        last_on = (link_ep if name == \"link\" else power_ep).filter(pl.col(\"onset_type\") == \"strict\").select(\n            \"ch\", pl.col(\"onset_t\").alias(f\"_{name}_last_onset\")).sort(f\"_{name}_last_onset\")\n        panel = panel.join_asof(last_on, left_on=\"_cutoff_t\", right_on=f\"_{name}_last_onset\", by=\"ch\",\n                                strategy=\"backward\").with_columns(\n            ((pl.col(\"_cutoff_t\") - pl.col(f\"_{name}_last_onset\")).dt.total_seconds() / 86400.0)\n            .alias(f\"f_days_since_{name}_onset\"))\n    panel = panel.sort(\"ch\", \"day\")\n\n    # факты будущих дней для меток\n    fact_cols = [\"n_events\", \"link_onset_strict\", \"link_onset_ambiguous\", \"link_onset_sustained\", \"link_onset_censored\",\n                 \"link_onset_transient\", \"power_onset_strict\", \"power_onset_ambiguous\", \"power_onset_sustained\",\n                 \"power_onset_censored\", \"power_onset_transient\", \"link_ambiguous_block\", \"power_ambiguous_block\",\n                 \"n_tech_like\", \"n_measurement\", \"gas_n\",\n                 \"gas_cross_n\", \"gas_cross_unknown_n\"]\n    # каждый наблюдённый день есть в гриде (k = 0), поэтому факты любого дня с событиями — в panel\n    facts = panel.select(\"ch\", \"day\", *fact_cols)\n    d1 = _shift_days(pl, facts, [\"link_onset_strict\", \"link_onset_ambiguous\", \"power_onset_strict\",\n                                 \"power_onset_ambiguous\", \"gas_cross_n\"], 1, \"_d1\")\n    d2 = _shift_days(pl, facts, fact_cols, 2, \"_d2\")\n    d3 = _shift_days(pl, facts, [\"n_events\"], 3, \"_d3\")\n    panel = panel.join(d1, on=[\"ch\", \"day\"], how=\"left\").join(d2, on=[\"ch\", \"day\"], how=\"left\") \\\n                 .join(d3, on=[\"ch\", \"day\"], how=\"left\")\n    fut_cols = [c for c in panel.columns if c.endswith((\"_d1\", \"_d2\", \"_d3\"))]\n    panel = panel.with_columns([pl.col(c).fill_null(0) for c in fut_cols])\n\n    in_data = (pl.col(\"day\") + pl.duration(days=3)) <= pl.lit(data_end_day)  # окно + L внутри данных\n    observed = in_data & ((pl.col(\"n_events_d2\") + pl.col(\"n_events_d3\")) > 0)\n\n    def onset_target(prefix: str, state_col: str, sustained: bool) -> tuple[Any, Any]:\n        strict = pl.col(f\"{prefix}_onset_strict_d2\")\n        amb = pl.col(f\"{prefix}_onset_ambiguous_d2\") + pl.col(f\"{prefix}_ambiguous_block_d2\")\n        eligible = pl.col(state_col) == \"O\"\n        if not sustained:\n            label = (pl.when(~eligible.fill_null(False)).then(None)\n                     .when(~in_data).then(None)\n                     .when(strict > 0).then(pl.lit(1))\n                     .when(amb > 0).then(None)\n                     .when(observed).then(pl.lit(0))\n                     .otherwise(None))\n            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit(\"no_state_history\"))\n                      .when(~eligible).then(pl.lit(\"state_not_ok_at_D\"))\n                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                      .when(strict > 0).then(pl.lit(\"positive\"))\n                      .when(amb > 0).then(pl.lit(\"ambiguous_onset\"))\n                      .when(observed).then(pl.lit(\"negative\"))\n                      .otherwise(pl.lit(\"unobserved_window\")))\n        else:\n            sus, cen, tra = (pl.col(f\"{prefix}_onset_sustained_d2\"), pl.col(f\"{prefix}_onset_censored_d2\"),\n                             pl.col(f\"{prefix}_onset_transient_d2\"))\n            label = (pl.when(~eligible.fill_null(False)).then(None)\n                     .when(~in_data).then(None)\n                     .when(sus > 0).then(pl.lit(1))\n                     .when(cen > 0).then(None)\n                     .when(amb > 0).then(None)\n                     .when(observed).then(pl.lit(0))  # включая только кратковременные (transient) onset\n                     .otherwise(None))\n            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit(\"no_state_history\"))\n                      .when(~eligible).then(pl.lit(\"state_not_ok_at_D\"))\n                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                      .when(sus > 0).then(pl.lit(\"positive\"))\n                      .when(cen > 0).then(pl.lit(\"censored_duration\"))\n                      .when(amb > 0).then(pl.lit(\"ambiguous_onset\"))\n                      .when(observed & (tra > 0)).then(pl.lit(\"negative_transient_only\"))\n                      .when(observed).then(pl.lit(\"negative\"))\n                      .otherwise(pl.lit(\"unobserved_window\")))\n        return label.cast(pl.Int8), reason\n\n    targets = {}\n    reasons = {}\n    for name, prefix, state, sus in ((\"target_t2a_link_onset\", \"link\", \"link_state\", False),\n                                     (\"target_t2b_link_sustained\", \"link\", \"link_state\", True),\n                                     (\"target_t2c_power_onset\", \"power\", \"power_state\", False),\n                                     (\"target_t2c_power_sustained\", \"power\", \"power_state\", True)):\n        lab, rea = onset_target(prefix, state, sus)\n        targets[name], reasons[name] = lab, rea\n\n    # T1a: технические/диапазонные значения после ≥ tech_clean_days чистых суток\n    clean = int(config[\"tech_clean_days\"])\n    hist = int(config[\"numeric_history_days\"])\n    panel = panel.with_columns(\n        pl.col(\"n_tech_like\").fill_null(0).rolling_sum_by(\"day\", window_size=f\"{clean}d\", closed=\"right\").over(\"ch\").alias(\"_tech_recent\"),\n        pl.col(\"n_measurement\").fill_null(0).rolling_sum_by(\"day\", window_size=f\"{hist}d\", closed=\"right\").over(\"ch\").alias(\"_meas_hist\"),\n    )\n    t1_elig = (pl.col(\"_meas_hist\") > 0) & (pl.col(\"_tech_recent\") == 0)\n    targets[\"target_t1a_tech_value\"] = (pl.when(~t1_elig | ~in_data).then(None)\n                                        .when(pl.col(\"n_tech_like_d2\") > 0).then(pl.lit(1))\n                                        .when(pl.col(\"n_measurement_d2\") > 0).then(pl.lit(0))\n                                        .otherwise(None)).cast(pl.Int8)\n    reasons[\"target_t1a_tech_value\"] = (pl.when(pl.col(\"_meas_hist\") == 0).then(pl.lit(\"no_measurements_30d\"))\n                                        .when(pl.col(\"_tech_recent\") > 0).then(pl.lit(\"recent_tech_value\"))\n                                        .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                                        .when(pl.col(\"n_tech_like_d2\") > 0).then(pl.lit(\"positive\"))\n                                        .when(pl.col(\"n_measurement_d2\") > 0).then(pl.lit(\"negative\"))\n                                        .otherwise(pl.lit(\"unobserved_window\")))\n\n    # T4: газ — строгое пересечение 1 % в окне; канал газовый, последнее показание на конец D < порога\n    thr = config[\"gas_threshold_percent\"]\n    gas_elig = (pl.col(\"f_gas_n_7d\") > 0) & (pl.col(\"f_gas_last\") < thr) & ~pl.col(\"_gas_last_mixed\")\n    targets[\"target_t4_gas_cross\"] = (pl.when(~gas_elig.fill_null(False) | ~in_data).then(None)\n                                      .when(pl.col(\"gas_cross_n_d2\") > 0).then(pl.lit(1))\n                                      .when(pl.col(\"gas_cross_unknown_n_d2\") > 0).then(None)\n                                      .when(pl.col(\"gas_n_d2\") > 0).then(pl.lit(0))\n                                      .otherwise(None)).cast(pl.Int8)\n    reasons[\"target_t4_gas_cross\"] = (pl.when(pl.col(\"f_gas_n_7d\").fill_null(0) == 0).then(pl.lit(\"not_gas_stream\"))\n                                      .when(pl.col(\"_gas_last_mixed\")).then(pl.lit(\"ambiguous_last_reading_at_D\"))\n                                      .when(pl.col(\"f_gas_last\") >= thr).then(pl.lit(\"above_threshold_at_D\"))\n                                      .when(~in_data).then(pl.lit(\"window_beyond_data\"))\n                                      .when(pl.col(\"gas_cross_n_d2\") > 0).then(pl.lit(\"positive\"))\n                                      .when(pl.col(\"gas_cross_unknown_n_d2\") > 0).then(pl.lit(\"unknown_previous_reading\"))\n                                      .when(pl.col(\"gas_n_d2\") > 0).then(pl.lit(\"negative\"))\n                                      .otherwise(pl.lit(\"unobserved_window\")))\n\n    panel = panel.with_columns(\n        **targets, **{f\"reason_{k[len('target_'):]}\": v for k, v in reasons.items()},\n        competing_link_onset_d1=(pl.col(\"link_onset_strict_d1\") + pl.col(\"link_onset_ambiguous_d1\") > 0),\n        competing_power_onset_d1=(pl.col(\"power_onset_strict_d1\") + pl.col(\"power_onset_ambiguous_d1\") > 0),\n        competing_gas_cross_d1=(pl.col(\"gas_cross_n_d1\") > 0),\n    )\n\n    # мета: тип, объект, псевдоним ключей; календарь\n    panel = panel.join(meta, on=\"ch\", how=\"left\").with_columns(\n        pl.col(\"day\").alias(\"d_cutoff_date\"),\n        (pl.col(\"day\") + pl.duration(days=2)).alias(\"d_target_start_date\"),\n        (pl.col(\"day\") + pl.duration(days=3)).alias(\"d_target_end_date_exclusive\"),\n        (pl.col(\"day\") + pl.duration(days=3) + pl.duration(hours=L)).alias(\"d_label_decision_end\"),\n        pl.col(\"day\").dt.year().alias(\"d_year\"),\n        pl.col(\"day\").dt.weekday().alias(\"d_weekday\"),\n        pl.col(\"day\").dt.month().alias(\"d_month\"),\n    )\n    keys = key_map(pl, panel[\"ch\"].unique().to_list(), \"d_channel_key\")\n    objs = key_map(pl, panel[\"ид_объект\"].drop_nulls().unique().to_list(), \"d_object_key\")\n    panel = (panel.join(keys, left_on=\"ch\", right_on=\"raw\", how=\"left\")\n             .join(objs, left_on=\"ид_объект\", right_on=\"raw\", how=\"left\"))\n\n    feature_cols = [c for c in panel.columns if c.startswith(\"f_\")] + BASE_FEATURES\n    target_cols = list(targets)\n    assert target_cols == TARGETS\n    keep = ([\"d_channel_key\", \"d_object_key\", \"d_cutoff_date\", \"d_target_start_date\", \"d_target_end_date_exclusive\",\n             \"d_label_decision_end\", \"d_year\", \"тип_датчика\", \"тип_инж_системы\"] + feature_cols + target_cols\n            + [f\"reason_{k[len('target_'):]}\" for k in target_cols]\n            + [\"competing_link_onset_d1\", \"competing_power_onset_d1\", \"competing_gas_cross_d1\"])\n    out = panel.select(keep).sort(\"d_channel_key\", \"d_cutoff_date\")\n\n    durations = pl.concat([\n        e.select(pl.lit(n).alias(\"kind\"), \"onset_type\", \"exact\", \"duration_s\", \"sustained\")\n        for n, e in ((\"link\", link_ep), (\"power\", power_ep))], how=\"vertical\")\n    return out, {\"rows\": out.height, \"channels\": out[\"d_channel_key\"].n_unique(), \"_episode_durations\": durations}\n\n\n# ---------------------------------------------------------------------------\n# 6. Аудит (только агрегаты)\n# ---------------------------------------------------------------------------\ndef final_audit(pl: Any, panel_lf: Any, durations: Any) -> dict[str, Any]:\n    def ep_stats(kind: str) -> dict[str, Any]:\n        ep = durations.filter(pl.col(\"kind\") == kind)\n        s = ep.filter(pl.col(\"onset_type\") == \"strict\")\n        dur = s.filter(pl.col(\"exact\"))[\"duration_s\"]\n        return {\n            \"episodes\": ep.height, \"strict_onsets\": s.height,\n            \"ambiguous_onsets\": int((ep[\"onset_type\"] == \"ambiguous\").sum()),\n            \"strict_exact\": dur.len(), \"strict_censored\": int((~s[\"exact\"]).sum()),\n            \"exact_duration_s_quantiles\": ({str(q): float(dur.quantile(q)) for q in (0.1, 0.5, 0.9, 0.99)} if dur.len() else None),\n            \"sustained_1\": int((s[\"sustained\"] == 1).sum()), \"sustained_0\": int((s[\"sustained\"] == 0).sum()),\n            \"sustained_null\": int(s[\"sustained\"].is_null().sum()),\n        }\n\n    head = panel_lf.select(pl.len().alias(\"rows\"), pl.col(\"d_channel_key\").n_unique().alias(\"channels\"),\n                           pl.col(\"d_cutoff_date\").min().alias(\"dmin\"), pl.col(\"d_cutoff_date\").max().alias(\"dmax\")).collect()\n    out: dict[str, Any] = {\"rows\": int(head[\"rows\"][0]), \"channels\": int(head[\"channels\"][0]),\n                           \"date_min\": str(head[\"dmin\"][0]), \"date_max\": str(head[\"dmax\"][0]),\n                           \"link_episodes\": ep_stats(\"link\"), \"power_episodes\": ep_stats(\"power\"), \"targets\": {}}\n    for t in TARGETS:\n        r = f\"reason_{t[len('target_'):]}\"\n        g = panel_lf.group_by(r).agg(pl.len()).collect()\n        by_year = panel_lf.group_by(\"d_year\").agg((pl.col(t) == 1).sum().alias(\"pos\"), (pl.col(t) == 0).sum().alias(\"neg\"),\n                                                  pl.col(t).null_count().alias(\"null\")).sort(\"d_year\").collect()\n        reasons = {k: int(v) for k, v in g.iter_rows()}\n        out[\"targets\"][t] = {\n            \"positive\": reasons.get(\"positive\", 0),\n            \"negative\": reasons.get(\"negative\", 0) + reasons.get(\"negative_transient_only\", 0),\n            \"null_reasons\": {k: v for k, v in reasons.items() if k not in (\"positive\", \"negative\", \"negative_transient_only\")},\n            \"negative_transient_only\": reasons.get(\"negative_transient_only\", 0),\n            \"by_year\": {str(y): {\"pos\": int(p), \"neg\": int(n), \"null\": int(nu)} for y, p, n, nu in by_year.iter_rows()},\n        }\n    return out\n\n\n# ---------------------------------------------------------------------------\n# 7. Поток по годам (для Kaggle): сырой журнал → части → панель\n# ---------------------------------------------------------------------------\ndef read_journal(pl: Any, path: Path) -> Any:\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    return pl.read_csv(path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\",\n                       columns=EVENT_COLUMNS).filter(pl.col(\"ид_события\") != \"ид_события\")\n\n\ndef read_catalogue(pl: Any, path: Path) -> Any:\n    cat = pl.read_csv(path, infer_schema_length=0, encoding=\"utf8-lossy\")\n    return cat.select([c for c in CATALOGUE_COLUMNS if c in cat.columns]).unique(\"ид_канала_данных\", keep=\"last\")\n\n\ndef process_year(pl: Any, raw: Any, tax: dict[str, Any], catalogue: Any, gas_carry: Any) -> dict[str, Any]:\n    ev = classify_events(pl, raw, tax, catalogue)\n    sec = second_table(pl, ev, tax)\n    daily = daily_aggregates(pl, ev, sec, tax)\n    gas_daily, gas_carry = gas_crossings(pl, ev, gas_carry, tax[\"numeric\"][\"gas_threshold_percent\"])\n    meta = ev.group_by(\"ch\").agg(pl.col(\"тип_датчика\").drop_nulls().last(), pl.col(\"тип_инж_системы\").drop_nulls().last(),\n                                 pl.col(\"ид_объект\").drop_nulls().last())\n    class_counts = {k: int(v) for k, v in ev.group_by(\"cls\").len().iter_rows()}\n    streams = tax[\"stream_mapping\"][\"streams\"]\n    mapped = {c for st in streams.values() for key in (\"fault\", \"ok\", \"unknown\") for c in st[key]}\n    stream_audit = {}\n    for name in streams:\n        g = sec.group_by(f\"{name}_cls\", f\"{name}_a_cause\").len()\n        stream_audit[name] = {f\"{st or 'none'}{'_' + cause if cause else ''}\": int(n) for st, cause, n in g.iter_rows()}\n    stream_audit[\"events_in_no_stream_by_class\"] = {k: v for k, v in class_counts.items() if k not in mapped}\n    # «Неопределен»: сколько каналов и какие классы соседствуют (агрегаты, порядок внутри секунды не используется)\n    evs = ev.select(\"ch\", \"t\", \"cls\").sort(\"ch\", \"t\")\n    evs = evs.with_columns(pl.col(\"cls\").shift(1).over(\"ch\").alias(\"prev\"), pl.col(\"cls\").shift(-1).over(\"ch\").alias(\"next\"))\n    svc = evs.filter(pl.col(\"cls\") == \"service\")\n    service_audit = {\"events\": svc.height, \"channels\": svc[\"ch\"].n_unique(),\n                     \"prev_class\": {str(k): int(v) for k, v in svc.group_by(\"prev\").len().sort(\"len\", descending=True).head(10).iter_rows()},\n                     \"next_class\": {str(k): int(v) for k, v in svc.group_by(\"next\").len().sort(\"len\", descending=True).head(10).iter_rows()}}\n    unknown_values = {k: int(v) for k, v in ev.filter(pl.col(\"cls\") == \"unknown_value\").group_by(\"v\").len()\n                      .sort(\"len\", descending=True).head(30).iter_rows()}\n    return {\n        \"daily\": daily, \"link_cp\": change_points(pl, sec, \"link_cls\"), \"power_cp\": change_points(pl, sec, \"power_cls\"),\n        \"gas_daily\": gas_daily, \"gas_carry\": gas_carry, \"meta\": meta, \"t_max\": ev[\"t\"].max(),\n        \"audit\": {\"events\": ev.height, \"channels\": ev[\"ch\"].n_unique(), \"seconds\": sec.height,\n                  \"multi_value_seconds\": int((sec[\"n_values\"] > 1).sum()),\n                  \"link_fault_seconds\": int((sec[\"link_cls\"] == \"F\").sum()),\n                  \"link_fault_seconds_with_other_value\": int(((sec[\"has_fault\"]) & (sec[\"n_values\"] > 1)).sum()),\n                  \"link_ambiguous_seconds\": int((sec[\"link_cls\"] == \"A\").sum()),\n                  \"class_counts\": class_counts, \"unknown_values_top30\": unknown_values,\n                  \"stream_mapping\": stream_audit, \"service_value\": service_audit},\n    }\n\n\ndef empty_gas_carry(pl: Any) -> Any:\n    return pl.DataFrame(schema={\"ch\": pl.String, \"t\": pl.Datetime(\"us\"), \"mn\": pl.Float64, \"mx\": pl.Float64})\n\n\ndef journal_max_time(pl: Any, path: Path) -> datetime:\n    \"\"\"Лёгкий предпроход: последний момент в файле (для общей границы данных).\"\"\"\n    t = pl.concat_str([\"дата\", \"время\"], separator=\" \").str.strptime(pl.Datetime(\"us\"), \"%Y-%m-%d %H:%M:%S\", strict=False)\n    return _scan_source(pl, path).select(t.max()).collect().item()\n\n\ndef _scan_source(pl: Any, path: Path) -> Any:\n    path = Path(path)\n    if path.suffix == \".parquet\":\n        return pl.scan_parquet(path)\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    return pl.scan_csv(path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\")\n\n\ndef scan_bucket(pl: Any, path: Path, bucket: int, n_buckets: int, channel_filter: Any = None) -> Any:\n    lf = _scan_source(pl, path).select(EVENT_COLUMNS).filter(pl.col(\"ид_события\") != \"ид_события\")\n    if n_buckets > 1:\n        lf = lf.filter(pl.col(\"ид_канала_данных\").hash(seed=20260923) % n_buckets == bucket)\n    if channel_filter is not None:\n        lf = lf.filter(channel_filter)\n    return lf.collect()\n\n\ndef _merge_audits(parts: list[dict[str, Any]]) -> dict[str, Any]:\n    out: dict[str, Any] = {}\n    for a in parts:\n        for k, v in a.items():\n            if isinstance(v, bool) or v is None:\n                out.setdefault(k, v)\n            elif isinstance(v, (int, float)):\n                out[k] = out.get(k, 0) + v\n            elif isinstance(v, dict):\n                out[k] = _merge_audits([out.get(k, {}), v])\n            else:\n                out.setdefault(k, v)\n    return out\n\n\ndef build_event_panel(pl: Any, journal_paths: list[Path], catalogue_path: Path, taxonomy_path: Path,\n                      config: dict[str, Any] | None = None, log: Any = print, n_buckets: int = 1,\n                      out_dir: Path | None = None, channel_filter: Any = None,\n                      data_end: datetime | None = None) -> tuple[Any, dict[str, Any]]:\n    \"\"\"Собирает панель по группам каналов (каналы независимы). При out_dir пишет части parquet и\n    возвращает (None, audit); иначе возвращает панель в памяти.\"\"\"\n    cfg = {**DEFAULT_CONFIG, **(config or {})}\n    tax = load_taxonomy(taxonomy_path)\n    cfg[\"gas_threshold_percent\"] = tax[\"numeric\"][\"gas_threshold_percent\"]\n    if not cfg[\"service_as_unknown\"]:\n        link = tax[\"stream_mapping\"][\"streams\"][\"link\"]\n        link[\"unknown\"] = [c for c in link[\"unknown\"] if c != \"service\"]\n    if cfg[\"power_ok_from_function\"]:\n        power = tax[\"stream_mapping\"][\"streams\"][\"power\"]\n        power[\"ok\"] = list(dict.fromkeys(power[\"ok\"] + [\"normal\", \"event_alarm\", \"numeric\"]))\n    cfg[\"stream_mapping_version\"] = tax[\"stream_mapping\"][\"version\"]\n    catalogue = read_catalogue(pl, catalogue_path)\n    # data_end задаётся явно на runtime (конец дня D): события после D в сборку не попадают (см. channel_filter)\n    t_max = data_end if data_end is not None else max(journal_max_time(pl, p) for p in journal_paths)\n    panels, part_paths, per_file_parts, ep_parts = [], [], [], []\n    for b in range(n_buckets):\n        carry = empty_gas_carry(pl)\n        parts: dict[str, list[Any]] = {\"daily\": [], \"link_cp\": [], \"power_cp\": [], \"gas_daily\": [], \"meta\": []}\n        per_file = {}\n        for path in journal_paths:\n            raw = scan_bucket(pl, Path(path), b, n_buckets, channel_filter)\n            if raw.is_empty():\n                continue\n            res = process_year(pl, raw, tax, catalogue, carry)\n            carry = res[\"gas_carry\"]\n            for k in parts:\n                parts[k].append(res[k])\n            per_file[Path(path).name] = res[\"audit\"]\n            del raw, res\n        if not parts[\"daily\"]:\n            continue\n        daily = pl.concat(parts[\"daily\"], how=\"vertical_relaxed\")\n        dup = daily.height - daily.select(\"ch\", \"day\").unique().height\n        if dup:\n            raise ValueError(f\"дневные агрегаты пересекаются между файлами: {dup} строк\")\n        link_cp = rle(pl, pl.concat(parts[\"link_cp\"], how=\"vertical\"))\n        power_cp = rle(pl, pl.concat(parts[\"power_cp\"], how=\"vertical\"))\n        gas_daily = pl.concat(parts[\"gas_daily\"], how=\"vertical\").group_by(\"ch\", \"day\").agg(pl.all().sum())\n        meta = pl.concat(parts[\"meta\"], how=\"vertical_relaxed\").group_by(\"ch\").agg(pl.all().drop_nulls().last())\n        panel, audit = build_panel_from_parts(pl, daily, link_cp, power_cp, gas_daily, meta, t_max, cfg)\n        per_file_parts.append({\"per_file\": per_file})\n        ep_parts.append(audit.pop(\"_episode_durations\"))\n        if out_dir is not None:\n            path = Path(out_dir) / f\"event_panel_v3_part{b:02d}.parquet\"\n            panel.write_parquet(path, compression=\"zstd\")\n            part_paths.append(path)\n            del panel\n        else:\n            panels.append(panel)\n        log(f\"группа каналов {b + 1}/{n_buckets}: строк {audit['rows']:,}, каналов {audit['channels']:,}\")\n    if out_dir is not None:\n        full = pl.scan_parquet(part_paths)\n    else:\n        full = pl.concat(panels, how=\"vertical_relaxed\").lazy()\n    audit = final_audit(pl, full, pl.concat(ep_parts, how=\"vertical\"))\n    audit[\"feature_columns\"] = [c for c in full.collect_schema().names() if c.startswith(\"f_\")] + BASE_FEATURES\n    audit[\"target_columns\"] = TARGETS\n    audit.update({\"schema_version\": SCHEMA_VERSION, \"config\": cfg, \"taxonomy_version\": tax[\"version\"],\n                  \"taxonomy_sha256\": tax[\"_sha256\"], \"data_end\": str(t_max), \"n_buckets\": n_buckets,\n                  \"per_file\": _merge_audits(per_file_parts)[\"per_file\"] if per_file_parts else {},\n                  \"part_files\": [p.name for p in part_paths]})\n    return (None if out_dir is not None else full.collect()), audit\n\n\ndef csv_to_parquet(pl: Any, csv_path: Path, out_path: Path) -> dict[str, Any]:\n    \"\"\"Сырой CSV → parquet только нужных колонок (все строки как строки, без интерпретации).\"\"\"\n    schema = {c: pl.String for c in EVENT_COLUMNS}\n    lf = pl.scan_csv(csv_path, schema_overrides=schema, infer_schema_length=0, encoding=\"utf8-lossy\").select(EVENT_COLUMNS)\n    lf.sink_parquet(out_path, compression=\"zstd\")\n    rows = pl.scan_parquet(out_path).select(pl.len()).collect().item()\n    return {\"rows\": int(rows), \"parquet_bytes\": Path(out_path).stat().st_size}\n\n\n# ---------------------------------------------------------------------------\n# 8. Самопроверки на синтетике (для notebook)\n# ---------------------------------------------------------------------------\ndef run_event_panel_self_tests(taxonomy_path: Path, work_dir: Path) -> dict[str, bool]:\n    import polars as pl\n    import random\n\n    work_dir = Path(work_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    t0 = datetime(2024, 3, 1)\n\n    def events(rows):\n        return pl.DataFrame({\"ид_события\": [str(i) for i in range(len(rows))],\n                             \"ид_канала_данных\": [r[0] for r in rows],\n                             \"дата\": [r[1].strftime(\"%Y-%m-%d\") for r in rows],\n                             \"время\": [r[1].strftime(\"%H:%M:%S\") for r in rows],\n                             \"тревожное\": [\"f\"] * len(rows), \"значение_датчика\": [r[2] for r in rows]})\n\n    def catalogue(chs, typ=\"Состояние насоса\"):\n        return pl.DataFrame({\"ид_канала_данных\": chs, \"тип_инж_системы\": [\"x\"] * len(chs),\n                             \"тип_датчика\": [typ] * len(chs), \"ид_объект\": [\"1\"] * len(chs)})\n\n    def build(name, rows, chs, **kw):\n        d = work_dir / name\n        d.mkdir(exist_ok=True)\n        events(rows).write_csv(d / \"j.csv\")\n        catalogue(chs).write_csv(d / \"c.csv\")\n        return build_event_panel(pl, [d / \"j.csv\"], d / \"c.csv\", taxonomy_path, log=lambda *a: None, **kw)\n\n    def row(panel, ch, day):\n        r = panel.filter((pl.col(\"d_channel_key\") == pseudo_key(ch)) & (pl.col(\"d_cutoff_date\") == (t0 + timedelta(days=day)).date()))\n        return r.row(0, named=True)\n\n    report: dict[str, bool] = {}\n    hb = [(\"A\", t0 + timedelta(days=d, hours=12), \"Норма\") for d in range(12)]\n    p, _ = build(\"onset\", hb + [(\"A\", t0 + timedelta(days=5, hours=13), \"Неисправен\"), (\"A\", t0 + timedelta(days=5, hours=14), \"Норма\")], [\"A\"])\n    report[\"strict_onset_in_window_is_positive\"] = row(p, \"A\", 3)[\"target_t2a_link_onset\"] == 1\n    report[\"onset_in_d1_is_competing_not_filtered\"] = row(p, \"A\", 4)[\"target_t2a_link_onset\"] == 0 and row(p, \"A\", 4)[\"competing_link_onset_d1\"]\n    t = t0 + timedelta(days=5, hours=13)\n    p, _ = build(\"amb\", hb + [(\"A\", t, \"Неисправен\"), (\"A\", t, \"Норма\")], [\"A\"])\n    r = row(p, \"A\", 3)  # конфликт в одной секунде → не onset и не 0, а null(ambiguous_onset)\n    report[\"same_second_conflict_is_null\"] = r[\"target_t2a_link_onset\"] is None and r[\"reason_t2a_link_onset\"] == \"ambiguous_onset\"\n    rnd = random.Random(1)\n    vals = [\"Норма\", \"Неисправен\", \"Обесточен\", \"Есть питание\", \"Выключен\", \"Неопределен\", \"0,05\", \"1,3\", \"-127\"]\n    rows = [(f\"C{c}\", t0 + timedelta(days=d, seconds=rnd.randint(0, 86399)), rnd.choice(vals))\n            for c in range(8) for d in range(30) for _ in range(rnd.randint(0, 5))]\n    chs = [f\"C{c}\" for c in range(8)]\n    base, _ = build(\"rand1\", rows, chs)\n    out = work_dir / \"rand_parts\"\n    out.mkdir(exist_ok=True)\n    build(\"rand2\", rows, chs, n_buckets=3, out_dir=out)\n    parts = pl.read_parquet(sorted(out.glob(\"*.parquet\")))\n    key = [\"d_channel_key\", \"d_cutoff_date\"]\n    report[\"channel_buckets_do_not_change_panel\"] = base.sort(key).equals(parts.sort(key))\n    report[\"no_raw_identifiers_in_panel\"] = not any(c in base.write_csv() for c in (\"C1,\", \"C2,\"))\n    return report\n\n\ndef label_changes(pl: Any, main: Any, variant: Any, targets: list[str] | None = None) -> dict[str, dict[str, int]]:\n    \"\"\"Сколько меток меняется между основной и sensitivity-версией (только агрегаты, по ключу строки).\"\"\"\n    k = [\"d_channel_key\", \"d_cutoff_date\"]\n    out: dict[str, dict[str, int]] = {}\n    for t in targets or TARGETS:\n        j = main.select(k + [t]).join(variant.select(k + [t]), on=k, how=\"full\", suffix=\"_v\", coalesce=True)\n        a = pl.col(t).cast(pl.String).fill_null(\"null\")\n        b = pl.col(f\"{t}_v\").cast(pl.String).fill_null(\"null\")\n        g = j.with_columns(a.alias(\"a\"), b.alias(\"b\")).filter(pl.col(\"a\") != pl.col(\"b\")).group_by(\"a\", \"b\").len()\n        out[t] = {f\"{x}->{y}\": int(n) for x, y, n in g.sort(\"a\", \"b\").iter_rows()}\n    return out\n",
"config/state_taxonomy_v3.json": "{\n  \"version\": \"3.1\",\n  \"description\": \"Классы значений значение_датчика для событийной панели v3. Не метки физических отказов. Источник: справочник_состояний.csv (организаторы) + словарь значений журнала; расхождения со справочником отмечены в notes.\",\n  \"text_classes\": {\n    \"link_fault\": [\n      \"Неисправен\"\n    ],\n    \"power_off\": [\n      \"Обесточен\"\n    ],\n    \"power_on\": [\n      \"Есть питание\",\n      \"Питание от сети\"\n    ],\n    \"power_degraded\": [\n      \"Питание от батарей\",\n      \"Батарея разряжена\",\n      \"Батарея неисправна\"\n    ],\n    \"device_off\": [\n      \"Отключено устройство\"\n    ],\n    \"service\": [\n      \"Неопределен\"\n    ],\n    \"undefined_measurement\": [\n      \"Не определено\"\n    ],\n    \"object_summary\": [\n      \"Много неисправных устройств\",\n      \"Устройства на объекте исправны\",\n      \"Работают все насосы АНС\",\n      \"Работают все насосы в АНС\",\n      \"Включены все насосы АНС\"\n    ],\n    \"security_mode\": [\n      \"На охране\",\n      \"Снято с охраны\"\n    ],\n    \"flood\": [\n      \"Затоплен\"\n    ],\n    \"event_alarm\": [\n      \"Обнаружено движение\",\n      \"Движение вверх\",\n      \"Движение вниз\",\n      \"Движение влево\",\n      \"Движение вправо\",\n      \"Обнаружен дым\",\n      \"Обнаружен газ\",\n      \"Не замкнут\",\n      \"Замкнут\",\n      \"Рычаг сдернут\",\n      \"Рычаг сдернут влево\",\n      \"Рычаг сдернут вправо\",\n      \"Оба рычага сдернуты\",\n      \"Вызов\",\n      \"Разговор\",\n      \"Температура выше 35ºC1\",\n      \"Температура выше 40ºC\",\n      \"Температура выше 27ºC\",\n      \"Температура ниже 3ºC\",\n      \"Температура ниже 3ºC1\"\n    ],\n    \"normal\": [\n      \"Норма\",\n      \"Движения нет\",\n      \"Дыма нет\",\n      \"Рычаг норма\",\n      \"Включен\",\n      \"Выключен\",\n      \"В норме от +3 до +40\",\n      \"В норме от +3 до +351\",\n      \"В норме от +3 до +27ºC\"\n    ]\n  },\n  \"numeric\": {\n    \"technical_value_candidates\": [\n      -127.0,\n      -100.0,\n      255.0,\n      -3276.0,\n      -3276.8,\n      327.68,\n      3276.7\n    ],\n    \"gas_sensor_type\": \"Газовый датчик\",\n    \"gas_out_of_range\": {\n      \"below\": 0.0,\n      \"above\": 100.0\n    },\n    \"gas_threshold_percent\": 1.0,\n    \"gas_warning_percent\": 5.0,\n    \"gas_band_low\": 0.1\n  },\n  \"date_like\": {\n    \"regex\": \"^\\\\d{2}\\\\.\\\\d{2}\\\\.\\\\d{4}\",\n    \"clock_1970_prefix\": \"01.01.1970\",\n    \"duplicate_timestamp_tolerance_seconds\": 5\n  },\n  \"notes\": [\n    \"Неисправен = «только факт потери связи с устройством» (FAQ организаторов); в справочнике есть только у набора 1 (КД Дверь), в журнале — у насосов, вентиляторов, дыма, газа, фаз.\",\n    \"Обесточен у «Состояния фазы» в справочнике отсутствует; у набора 10 (Датчик дыма) помечен тревожное=false.\",\n    \"−100, 255 — примеры технических кодов производителей из FAQ, не универсальные пороги; кандидаты проверяются агрегатами на train.\",\n    \"01.01.1970 — ошибка часов/формата (FAQ: «скорее всего … сбилась дата»), не метка отказа измерителя; даты, совпадающие со временем события, — дубль timestamp.\",\n    \"Связь: O — только сообщения функции самого канала (норма/событие/число); питание, охрана, затопление, служебные — не O связи.\",\n    \"Питание: O — только явные «Есть питание»/«Питание от сети»; «Питание от батарей» нейтрально; во время «Неисправен» питание неизвестно (U).\",\n    \"«Неопределен» в основной версии — U для связи (не O и не восстановление); sensitivity-версия считает его нейтральным (GPT 16/Q1).\",\n    \"Коды T1a (−127, −100, 255, −3276, −3276.8, 327.68, 3276.7) — фиксированное правило из FAQ и данных 2019 до любой валидации; не означают физический отказ датчика.\"\n  ],\n  \"stream_mapping\": {\n    \"version\": \"1.0\",\n    \"note\": \"stream_id = (канал, класс сигнала). F — отказ потока, O — однозначное рабочее состояние того же потока. F+O в одной секунде → A (conflict). U-значение: при unknown_overrides=true делает секунду A всегда, иначе — только если в секунде нет F/O этого потока. Значения других классов поток не меняют (GPT 16: O другого класса не закрывает F связи; «Обесточен» не O связи).\",\n    \"streams\": {\n      \"link\": {\n        \"fault\": [\n          \"link_fault\"\n        ],\n        \"ok\": [\n          \"normal\",\n          \"event_alarm\",\n          \"numeric\"\n        ],\n        \"unknown\": [\n          \"service\"\n        ],\n        \"unknown_overrides\": true\n      },\n      \"power\": {\n        \"fault\": [\n          \"power_off\"\n        ],\n        \"ok\": [\n          \"power_on\"\n        ],\n        \"unknown\": [\n          \"link_fault\"\n        ],\n        \"unknown_overrides\": false\n      }\n    },\n    \"sensitivity\": {\n      \"service_neutral\": \"link.unknown без service: «Неопределен» не меняет состояние связи\"\n    }\n  }\n}\n"
}


In [ ]:
import sys, tempfile
from pathlib import Path
CODE_DIR = Path(tempfile.mkdtemp(prefix="ldt16_code_"))
for name, text in SOURCES.items():
    p = CODE_DIR / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
sys.path.insert(0, str(CODE_DIR))
os.environ["LDT_CODE_DIR"] = str(CODE_DIR)
import gas_bundle_v3 as gb
import target_models_v3_runtime as rt
print("код записан:", ", ".join(sorted(SOURCES)))


## Самопроверки на синтетике

Оба рецепта, приёмка bundle, остановка при нехватке положительных, отсутствие ключей каналов в выходах.


In [ ]:
SELF_TESTS = gb.run_export_self_tests(Path(tempfile.mkdtemp()))
for name, ok in SELF_TESTS.items():
    print("OK  " if ok else "FAIL", name)
assert all(SELF_TESTS.values()), "Самопроверки не пройдены"


## Единственный прогон: обучение, audit, bundle, приёмка


In [ ]:
CONFIG = gb.make_config_export(MODE, RECIPE)
try:
    RESULT = gb.run_export_v3(CONFIG)
except gb.RecipeStop as stop:
    print("ОСТАНОВКА ДО ОБУЧЕНИЯ:", stop)
    raise
except gb.ReleaseBlocked as blocked:
    print("RELEASE GATE НЕ ПРОЙДЕН — bundle не выдаётся:", blocked)
    raise
print()
print("Статус:", RESULT["status"], "| время:", RESULT["runtime_s"], "с")
print("Сверка с 15:", RESULT["reproduction_15"])
assert RESULT["acceptance"]["passed"], "Приёмка bundle не пройдена — bundle не передавать"


## Сводка


In [ ]:
from IPython.display import Markdown, display
OUT = Path(CONFIG["output_dir"])
display(Markdown((OUT / "summary_export_16_v3_ru.md").read_text(encoding="utf-8")))


In [ ]:
print("Файлы для backend и команды:")
for name in ("gas_cross_v3_bundle.zip", "gas_cross_v3_bundle.zip.sha256", "results_export_16_v3.json", "summary_export_16_v3_ru.md"):
    p = OUT / name
    print("-", name, f"({p.stat().st_size / 1024:.0f} КБ)" if p.exists() else "— НЕ СОЗДАН")
print("Построчные прогнозы и идентификаторы не сохраняются.")
